In [1]:
# Imports for the modeling notebook.
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import bioDraws, log, Elem
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
import pandas as pd
import biogeme.database as db
import pickle


In [2]:
# Load the cleaned dataset produced by `loading_and_cleaning_data.ipynb`.
# low_memory=False suppresses the mixed-type DtypeWarning on the wide CSV.
df = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)


## Sanity check and preparing the databases

In [3]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection','vehicle_type_3'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

continuous_vars = [
    'age',
    'Number of passengers',
    'number of involved vehicles',
    'vma',
    'age_2',
    'age_opposite_mean'
]

df_centered = df.copy()
# cellule 3, avant le centrage
df_centered['vma_raw'] = df['vma']
df_centered['Number of passengers raw'] = df['Number of passengers']

# Vitesse limite en binaire : 50 km/h ou plus contre moins de 50.
#
# Les 459 `vma = -1` sont le code « non renseigne » du BAAC, pas une vitesse.
# Les ranger dans « moins de 50 » fabriquerait une modalite fausse et leur
# pretterait l'effet des zones apaisees ; ils ont donc leur propre indicatrice,
# comme l'age du tiers manquant. Le calcul se fait sur `vma` AVANT centrage.
df_centered['speed_limit_unknown'] = (df['vma'] <= 0).astype(int)
df_centered['speed_limit_50_plus'] = (df['vma'] >= 50).astype(int)
df_centered['speed_limit_30_plus'] = (df['vma'] >= 30).astype(int)
# La reference est donc « moins de 50 km/h, vitesse connue ». Pour prendre au
# contraire « 50 km/h ou plus » comme reference -- la modalite la plus
# frequente --, il suffit d'entrer `speed_limit_below_50` a la place.
df_centered['speed_limit_below_50'] = (
    (df['vma'] > 0) & (df['vma'] < 50)
).astype(int)

# La vitesse reste CONTINUE dans les modeles, mais debarrassee de son code
# manquant : `vma = -1` passe en NaN avant le centrage, comme l'age du tiers.
# Sans cela la moyenne est tiree vers le bas et toute la variable est decalee.
# `speed_limit_unknown` prend le relais dans les utilites, selon la methode
# d'ajustement par variable indicatrice decrite dans le manuscrit.
df_centered['vma'] = df_centered['vma'].mask(df_centered['vma'] <= 0)

df_centered['speed_limit_below_30'] = (
    (df['vma'] > 0) & (df['vma'] < 30)
).astype(int)

# « Pas d'opposant » n'est pas un age. Le pipeline code ce cas 0 (ligne 1230 de
# `loading_and_cleaning_data.ipynb`, pour les accidents a un seul vehicule), et
# ces 2 210 zeros tirent la moyenne de 41,6 a 35,3 ans.
#
# Le masque porte sur le CRITERE, pas sur la valeur : trois accidents pietons
# ont un opposant reellement age de 0 an -- un nourrisson en poussette, dont la
# ligne miroir du meme accident confirme l'age. Masquer « == 0 » les compterait
# a tort comme des tiers non identifies.
df_centered['age_opposite_mean'] = df_centered['age_opposite_mean'].mask(
    df_centered['vehicle_type_2'] == 'No other vehicle'
)

df_centered[continuous_vars] = (
    df_centered[continuous_vars]
    - df_centered[continuous_vars].mean()
)


df_non_dummies = df_centered[['age','severity','Number of passengers','number of involved vehicles','vma','vma_raw','age_2','catu', 'Num_Acc','age_opposite_mean','speed_limit_50_plus','speed_limit_below_50', 'speed_limit_30_plus','speed_limit_below_30','speed_limit_unknown']]




In [4]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


# On garde les observations sans tiers identifie au lieu de les supprimer :
# la variable est mise a 0 -- soit la moyenne, puisque les continues sont
# centrees -- et l'indicatrice ci-dessous absorbe le niveau propre de ce groupe.
# `beta_age_opposite_mean` reste donc identifie sur les seules lignes renseignees.
df_non_dummies['age_opposite_missing'] = (
    df_non_dummies['age_opposite_mean'].isna().astype(int)
)
df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(0.0)

# Meme traitement pour la vitesse limite : une fois centree, imputer par 0
# revient a imputer par la moyenne des vitesses RENSEIGNEES. L'indicatrice
# `speed_limit_unknown`, creee plus haut, porte l'ecart propre a ces
# observations -- c'est l'ajustement par variable indicatrice du manuscrit.
# Biogeme refuserait de toute facon un NaN dans la base.
df_non_dummies['vma'] = df_non_dummies['vma'].fillna(0.0)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [5]:
## First model
df_motorized_vehicles=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_motorized_vehicles=df_motorized_vehicles.loc[df_motorized_vehicles['catu'].isin([1,2])]
database_motorized_vehicles = db.Database('database_motorized_vehicles', df_motorized_vehicles)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
# L'unique observation mortelle est CONSERVEE : le modele regroupe deces et
# blessure via `harmed`, donc un deces isole n'a plus besoin d'etre identifie
# separement. La supprimer ferait aussi diverger cet echantillon de celui des
# statistiques descriptives, qui le conservent.
df_mmv = df_mmv.copy()

database_mmv= db.Database('database_mmv',df_mmv)

# Echantillon a TIERS IDENTIFIE, celui sur lequel les modeles MMV et pieton sont
# estimes. Les lignes ecartees sont celles dont le tiers a quitte les lieux : son
# vehicule, sa manoeuvre et son point de choc sont notes, mais ni son age ni son
# sexe. Dans 88 des 91 cas du segment MMV ce tiers est indemne, si bien que
# l'accident n'est au BAAC que parce que le sujet, lui, est blesse -- l'issue
# « indemne » n'y est pas rare, elle est inobservable. Ces lignes n'apportent
# donc aucune information sur la gravite, et une indicatrice de manquant y
# separerait parfaitement l'issue.
df_mmv_identified = df_mmv.loc[df_mmv['age_opposite_missing'] == 0].copy()
database_mmv_identified = db.Database('database_mmv_identified', df_mmv_identified)
print(f'MMV      : {len(df_mmv)} observations, {len(df_mmv_identified)} a tiers identifie')


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.copy()

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

df_pedestrian_identified = df_pedestrian.loc[
    df_pedestrian['age_opposite_missing'] == 0].copy()
database_pedestrian_identified = db.Database('database_pedestrian_identified',
                                             df_pedestrian_identified)
print(f'Pietons  : {len(df_pedestrian)} observations, '
      f'{len(df_pedestrian_identified)} a tiers identifie')

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


MMV      : 1282 observations, 1191 a tiers identifie
Pietons  : 2792 observations, 2790 a tiers identifie


## Variables and Betas

In [6]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)

# Reponse binaire : 1 indemne, 2 blesse ou tue.
#
# Certains segments ne comptent pas assez de deces pour identifier un
# parametre qui leur soit propre -- 9 chez les pietons, 1 dans les collisions
# engin-engin. Plutot que d'ecarter ces observations, on regroupe les deux
# issues nefastes. La fusion porte sur la VARIABLE EXPLIQUEE, jamais sur les
# utilites : donner la meme utilite a deux alternatives ne les fusionne pas,
# elle contraint leurs probabilites a etre egales, ce que les donnees
# dementent violemment.
harmed = (severity > 1) + 1
availability_harmed = {1: 1, 2: 1}


In [7]:
# --- Where each model writes its results --------------------------------------
# Biogeme writes `<modelName>.html`, `<modelName>.pickle` and the iterations file
# `__<modelName>.iter` in the current working directory, and exposes no parameter
# to redirect them. Putting a path in `modelName` is not an option: it would also
# become the name displayed inside the results, and it would turn the iterations
# file into an invalid path. Changing directory around the call is the only way.

import os
from contextlib import contextmanager
from pathlib import Path

RESULTS_ROOT = Path('results')
RESULTS_DIRECTORIES = {
    'motorized_vehicles': RESULTS_ROOT / 'model_motorized_vehicles',
    'mmv': RESULTS_ROOT / 'model_mmv',
    'pedestrian': RESULTS_ROOT / 'model_pedestrian',
    'single_vehicle': RESULTS_ROOT / 'model_single_vehicle',
}
for directory in RESULTS_DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)


@contextmanager
def _in_results_directory(segment):
    """Temporarily make `segment`'s directory the working directory.

    The Biogeme object must already be built when this is used: it reads
    `biogeme.toml` from the working directory at construction time, so building
    it inside the sub-directory would silently fall back to default parameters.
    """
    previous = Path.cwd()
    os.chdir(RESULTS_DIRECTORIES[segment])
    try:
        yield
    finally:
        os.chdir(previous)


def save_results(results, segment, model_name):
    """Write an already-estimated model's HTML and pickle into `segment`'s directory.

    Used for the models that are estimated through a helper rather than directly,
    which switch Biogeme's own output off so that intermediate fits do not
    produce files. Any previous file of the same name is removed first: Biogeme
    otherwise appends a ~00, ~01, ... suffix at every run, which is what filled
    the repository root with hundreds of files.
    """
    results.data.modelName = model_name
    with _in_results_directory(segment):
        for extension in ('html', 'pickle'):
            Path(f'{model_name}.{extension}').unlink(missing_ok=True)
        results.write_html(True)
        results.write_pickle()
    return results


def estimate_in(segment, the_biogeme, **kwargs):
    """Estimate a model, writing its HTML/pickle output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.estimate(**kwargs)


def validate_in(segment, the_biogeme, estimation_results, validation_data):
    """Out-of-sample validation, writing its output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.validate(estimation_results, validation_data)

## Model for car crashes

### Baseline (constants only)

We first estimate the constants-only model. Its log-likelihood is the
denominator used for the rho-square statistics reported below and for the
likelihood-ratio test (see the LR-test section).


In [8]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_motorized_vehicles'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_motorized_vehicles, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = estimate_in('motorized_vehicles', model_cst_car)



### Full mixed-logit specification


In [9]:
# `Maneuver_2` / `vehicle_type_2` decrivent le premier tiers, `_3` le second.
# Les sommer donnerait 2 quand les deux tiers partagent la caracteristique : on
# veut une indicatrice « au moins un des tiers », donc un OU logique. Sur ce
# segment, 35 collisions ont un vehicule motorise leger en tiers 3 sans l'avoir
# en tiers 2, et une seule les deux.
def any_of(*terms):
    """Indicatrice 0/1 : au moins un des termes est non nul."""
    total = terms[0]
    for term in terms[1:]:
        total = total + term
    return total > 0


v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I +beta_age_I*age
      + beta_user_category_passenger_I * user_category_passenger
     + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I * point_of_impact_back
      + beta_vehicle_type_2_light_motorized_vehicle_I * any_of(vehicle_type_2_light_motorized_vehicle,
                                                             vehicle_type_3_light_motorized_vehicle)
    # + beta_maneuver_2_overtaking_I * any_of(maneuver_2_overtaking, maneuver_3_overtaking)
      + beta_maneuver_without_change_of_direction_I * maneuver_without_change_of_direction
    #  + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
     + beta_intersection_no_intersection_I * (intersection_no_intersection)
)

v_fatality = (  constant_F
     
        + beta_age_F * age

        # Interaction retenue : M0 -> M2 significatif (LR = 10.90, p = 0.001)
        # et M2 -> M3 non significatif (LR = 0.10, p = 0.76), donc ajouter
        # une pente commune par-dessus n'apporte rien. La vitesse n'entre
        # donc QUE hors intersection.
        + beta_vma_F * vma
       # + beta_speed_limit_unknown_F * speed_limit_unknown
                      + beta_vehicle_type_2_large_motorized_vehicle_F * any_of(vehicle_type_2_large_motorized_vehicle,
                                                                vehicle_type_3_large_motorized_vehicle)
                                                           
        + beta_lighting_conditions_night_with_street_lightings_on_F * (lighting_conditions_night_with_street_lightings_on + lighting_conditions_night_without_street_lightings)
        + beta_maneuver_2_turning_right_F * any_of(maneuver_2_turning_right, maneuver_3_turning_right)
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
       # + beta_lighting_conditions_night_without_street_lightings_F* lighting_conditions_night_without_street_lightings
   #    + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
    # + beta_intersection_no_intersection_F  * intersection_no_intersection
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [10]:


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] 
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


logprob = log((prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_motorized_vehicles,logprob)
model_car.modelName = "logit_car_crashes"

# Estimate the parameters. 
results_ml_motorized_vehicles = estimate_in('motorized_vehicles', model_car)


In [11]:
results_ml_motorized_vehicles.get_estimated_parameters().round(4)


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-0.8898,0.4876,-1.8249,0.0680
beta_age_F,0.0544,0.0093,5.8653,0.0000
beta_age_I,0.0158,0.0046,3.4622,0.0005
beta_gender_female_I,0.6304,0.1351,4.6681,0.0000
beta_intersection_no_intersection_I,0.2491,0.1244,2.0028,0.0452
beta_lighting_conditions_night_with_street_lightings_on_F,0.8709,0.2935,2.9674,0.0030
beta_maneuver_2_turning_right_F,1.0909,0.2995,3.6428,0.0003
beta_maneuver_without_change_of_direction_I,0.4106,0.1157,3.5496,0.0004
beta_number_of_involved_vehicles_I,-0.9584,0.2043,-4.6918,0.0000
beta_point_of_impact_back_I,-0.3978,0.1518,-2.6203,0.0088


### Ordered-probit counterpart

An ordered model has a single latent index, so it cannot give a variable to one
severity level only. The index below is therefore the **union** of the
regressors of the injury and fatality utilities -- fifteen terms, against the
twenty coefficients the MNL spends on the same information.

No constant enters the index: the threshold plays that role.


In [12]:
# --- Car crashes: ordered-probit counterpart of the MNL above -----------------

car_index_terms = {
    'gender_female': gender_female,
    'age': age,
    'user_category_passenger': user_category_passenger,
   # 'impact_back_bike': point_of_impact_back * (vehicle_bike + vehicle_e_bike),
   # 'impact_back_epmd': point_of_impact_back * vehicle_e_pmd,
    'light_motorized_vehicle': vehicle_type_2_light_motorized_vehicle,
    'large_motorized_vehicle': vehicle_type_2_large_motorized_vehicle,
#    'overtaking': maneuver_2_overtaking,
#    'no_change_of_direction_bike': maneuver_without_change_of_direction * vehicle_bike,
    'female_driver_passenger': user_category_passenger * gender_driver_female,
#    'at_intersection': (intersection_no_intersection == 0),
    'vma_no_intersection': vma * intersection_no_intersection,
   # 'night_street_lightings_on': lighting_conditions_night_with_street_lightings_on,
#    'night_no_street_lightings': lighting_conditions_night_without_street_lightings,
    'turning_right': maneuver_2_turning_right,
#    'on_cycle_facility': accident_location_on_cycle_facility,
}

index_car = None
for name, term in car_index_terms.items():
    contribution = Beta(f'beta_{name}_car_probit', 0, None, None, 0) * term
    index_car = contribution if index_car is None else index_car + contribution

model_name = 'ordered_probit_motorized_vehicles'
the_proba = ordered_probit(
    continuous_value=index_car,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=Beta('tau_1_car_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, severity))
model_car_probit = bio.BIOGEME(database_motorized_vehicles, logprob)
model_car_probit.modelName = model_name
results_car_probit = estimate_in('motorized_vehicles', model_car_probit)

print(f'MNL            : LL={results_ml_motorized_vehicles.data.logLike:9.3f}  '
      f'K={results_ml_motorized_vehicles.data.nparam:3d}  '
      f'AIC={results_ml_motorized_vehicles.data.akaike:8.2f}')
print(f'ordered probit : LL={results_car_probit.data.logLike:9.3f}  '
      f'K={results_car_probit.data.nparam:3d}  '
      f'AIC={results_car_probit.data.akaike:8.2f}')
results_car_probit.get_estimated_parameters().round(4)


MNL            : LL=-1450.167  K= 16  AIC= 2932.33
ordered probit : LL=-1498.533  K= 10  AIC= 3017.07


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_car_probit,0.0085,0.0018,4.6234,0.0000
beta_female_driver_passenger_car_probit,-0.4525,0.2269,-1.9940,0.0462
beta_gender_female_car_probit,0.1756,0.0515,3.4082,0.0007
beta_large_motorized_vehicle_car_probit,0.6118,0.1619,3.7781,0.0002
beta_light_motorized_vehicle_car_probit,-1.0888,0.0582,-18.6980,0.0000
beta_turning_right_car_probit,0.1340,0.0787,1.7029,0.0886
beta_user_category_passenger_car_probit,-0.8750,0.1516,-5.7718,0.0000
beta_vma_no_intersection_car_probit,0.0122,0.0048,2.5122,0.0120
tau_1_car_probit,-2.1248,0.0417,-50.9514,0.0000
tau_1_car_probit_diff_2,4.7165,0.0676,69.7551,0.0000


## MMV

### Baseline (constants only)


In [13]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [14]:
model_name = 'InitialModel_mmv'

# La disponibilite, pas les utilites : les deux alternatives sont toujours
# offertes. L'ancienne version passait {1: 0, 2: constant_I}, ce qui rendait
# l'alternative 1 indisponible -- d'ou l'avertissement "chosen alternative
# is not available" sur toutes les lignes sans blessure -- et faisait d'un
# parametre estime une disponibilite.
availability1 = {1: 1, 2: 1}

logprob_2 = models.loglogit(U, availability_harmed, harmed)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv_identified, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = estimate_in('mmv', model_cst_mmv)



### Full logit specification


In [15]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female + gender_3_male)
    + beta_point_of_impact_back_I          * point_of_impact_back
   # + beta_point_of_impact_back_I_epmd     * point_of_impact_back * vehicle_e_pmd
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
    + beta_maneuver_swerving_I             * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                         * age_opposite_mean
    #+ beta_vehicle_2_e_pmd_I               * any_of(vehicle_2_e_pmd + vehicle_3_e_pmd)
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [16]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability_harmed, harmed)
model_mmv = bio.BIOGEME(database_mmv_identified, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv = estimate_in('mmv', model_mmv)
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.026611,0.004733,-5.622755,1.879362e-08
beta_age_I,0.032443,0.005120,6.337035,2.342291e-10
beta_gender_2_female_I,-0.909218,0.139959,-6.496323,8.230705e-11
beta_gender_female_I,1.076689,0.168264,6.398810,1.565927e-10
beta_maneuver_swerving_I,-0.501510,0.188625,-2.658769,7.842673e-03
beta_maneuver_turning_left_I,-1.355752,0.317112,-4.275315,1.908674e-05
beta_point_of_impact_back_I,-0.885925,0.211546,-4.187869,2.815864e-05
beta_surface_condition_wet_I,-0.542370,0.229579,-2.362451,1.815455e-02
constant_I,0.778234,0.097647,7.969882,1.554312e-15


In [17]:
# --- MMV : le tiers non identifie, trois traitements ------------------------
# Constat : les 91 observations sans tiers identifie sont TOUTES blessees ou
# tuees (91 contre 0). Deux lectures possibles, et elles n'ont pas les memes
# consequences.
#
#   MAR      : ces accidents ressemblent aux autres a covariables egales, et
#              leur taux de blessure eleve est un hasard d'echantillonnage.
#   STRUCTUREL : un accident sans blesse dont le tiers a fui n'est tout
#              simplement pas enregistre. L'issue « indemne » n'est pas rare,
#              elle est IMPOSSIBLE dans ces lignes.
#
# On estime donc trois modeles, meme specification a chaque fois :
#
#   A  exclusion       : les 91 lignes sont retirees.
#   B1 melange fini    : elles sont gardees, le genre du tiers -- inconnu -- est
#                        somme sur ses deux modalites, ponderees par un petit
#                        logit pi(genre | covariables observees) estime
#                        CONJOINTEMENT. Hypothese MAR.
#   B2 disponibilite   : elles sont gardees, mais l'alternative « indemne » est
#                        declaree indisponible. Hypothese structurelle.
#
# B1 est un melange et non une integrale : la covariable manquante est discrete,
# une somme sur ses modalites suffit, donc pas de simulation.
#
# Aucun des trois n'est comparable aux autres par l'AIC -- echantillons ou
# variables expliquees differents. Ce qui se compare, ce sont LES COEFFICIENTS
# DE GRAVITE : s'ils bougent peu, le traitement du tiers manquant n'influence
# pas les conclusions, et c'est tout ce qu'on cherche a savoir.

from biogeme.expressions import Elem, Numeric, exp, log

identified = 1 - age_opposite_missing

# Quand le tiers n'est pas identifie, DEUX de ses caracteristiques manquent
# ensemble : son genre, discret, et son age, continu. Pour rester sur un
# MELANGE FINI -- une somme, sans simulation -- l'age est discretise en trois
# classes, representees par leur moyenne observee. La somme sur (genre, classe)
# compte donc six etats.
#
# C'est une quadrature dont les noeuds et les poids sont lus dans les donnees
# plutot que postules. L'approximation porte sur la finesse du decoupage, pas
# sur un tirage aleatoire : le resultat est deterministe et reproductible.
AGE_CLASSES = 3

_observed_age_2 = df_mmv.loc[df_mmv['age_opposite_missing'] == 0, 'age_opposite_mean']
_class_index = pd.qcut(_observed_age_2, AGE_CLASSES, labels=False)
AGE_CLASS_VALUES = [float(_observed_age_2[_class_index == b].mean())
                    for b in range(AGE_CLASSES)]
print('classes d\'age du tiers (variable centree) :',
      [round(v, 2) for v in AGE_CLASS_VALUES])

# La classe observee de chaque ligne, pour ancrer les poids sur les cas connus.
df_mmv_mixture = df_mmv.copy()
df_mmv_mixture['age_2_class'] = (
    pd.qcut(df_mmv_mixture['age_opposite_mean'].where(
        df_mmv_mixture['age_opposite_missing'] == 0),
        AGE_CLASSES, labels=False).fillna(0).astype(int))
age_2_class = Variable('age_2_class')


def severity_utility(female_state, age_2_value):
    """Utilite de blessure, genre et age du tiers etant fixes."""
    return (
          constant_I
        + beta_gender_female_I              * gender_female
        + beta_gender_2_female_I            * Numeric(female_state)
        + beta_point_of_impact_back_I_bike  * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
      #  + beta_point_of_impact_back_I_epmd  * point_of_impact_back * vehicle_e_pmd
        + beta_surface_condition_wet_I      * surface_condition_wet
        + beta_age_I                        * age
        + beta_maneuver_swerving_I          * maneuver_swerving
        + beta_maneuver_turning_left_I      * maneuver_turning_left
        + beta_age_2_I                      * age_2_value
    )


# P(blesse | ...) selon que l'age du tiers est observe ou tire.
choice_probability = {
    k: models.logit({1: 0, 2: severity_utility(k, age_opposite_mean)},
                    availability_harmed, harmed)
    for k in (0, 1)
}
# Une probabilite par etat (genre, classe d'age).
choice_probability_state = {
    (k, b): models.logit({1: 0, 2: severity_utility(k, Numeric(AGE_CLASS_VALUES[b]))},
                         availability_harmed, harmed)
    for k in (0, 1) for b in range(AGE_CLASSES)
}

# --- A. Exclusion -------------------------------------------------------------
data_identified = df_mmv[df_mmv['age_opposite_missing'] == 0].reset_index(drop=True)
database_mmv_identified = db.Database('mmv_identified', data_identified)

loglike_a = log(Elem(choice_probability, gender_2_female))  # age toujours observe ici
model_a = bio.BIOGEME(database_mmv_identified, loglike_a,
                      generate_html=True, generate_pickle=False)
model_a.modelName = 'mmv_a_exclusion'
results_a = estimate_in('mmv', model_a)

# --- B1. Melange fini ---------------------------------------------------------
# pi(genre du tiers), reduit a sa CONSTANTE. Des covariables observees avaient
# ete essayees -- age du sujet, type de son vehicule -- et n'expliquent rien
# (t = -1.5 et -0.3) : le genre du tiers ne se predit pas depuis les
# caracteristiques du sujet. La gravite, elle, n'a rien a faire ici : elle est
# deja dans le modele de choix, l'ajouter compterait deux fois la meme
# information.
#
# Dans B2, ce terme n'influence de toute facon aucun coefficient de gravite : la
# vraisemblance des lignes identifiees se separe en log P + log pi + log theta,
# et les trois blocs ne partagent aucun parametre. pi ne sert plus qu'a decrire
# la repartition des tiers.
pi_constant = Beta('pi_constant', 0, None, None, 0)

pi_female = exp(pi_constant) / (1 + exp(pi_constant))
pi = {0: 1 - pi_female, 1: pi_female}

# Poids des classes d'age, estimes eux aussi. Independance supposee entre le
# genre et la classe d'age du tiers, a covariables egales -- a declarer.
tau = {b: Beta(f'tau_age2_{b}', 0, None, None, 0) for b in range(1, AGE_CLASSES)}
tau[0] = Numeric(0)
_tau_denominator = sum(exp(tau[b]) for b in range(AGE_CLASSES))
theta = {b: exp(tau[b]) / _tau_denominator for b in range(AGE_CLASSES)}

# Ligne identifiee : le genre ET la classe d'age sont connus, et leurs poids
# entrent dans la vraisemblance -- c'est ce qui estime pi et theta sur les
# 1 191 cas observes. Sans ces termes, les poids ne seraient identifies que par
# les 91 lignes manquantes, toutes blessees.
likelihood_identified = (Elem({k: choice_probability[k] * pi[k] for k in (0, 1)},
                              gender_2_female)
                         * Elem(theta, age_2_class))

# Ligne non identifiee : somme sur les six etats.
likelihood_missing = sum(choice_probability_state[(k, b)] * pi[k] * theta[b]
                         for k in (0, 1) for b in range(AGE_CLASSES))

loglike_b1 = log(Elem({0: likelihood_missing, 1: likelihood_identified},
                      identified))
model_b1 = bio.BIOGEME(db.Database('mmv_b1', df_mmv_mixture), loglike_b1,
                       generate_html=True, generate_pickle=False)
model_b1.modelName = 'mmv_b1_melange'
results_b1 = estimate_in('mmv', model_b1)


# --- Comparaison des coefficients de gravite ----------------------------------
severity_betas = [name for name in results_a.get_beta_values()
                  if name.startswith(('beta_', 'constant'))]
comparison = pd.DataFrame({
    'A exclusion': results_a.get_beta_values(),
    'B1 melange': results_b1.get_beta_values(),
}).loc[severity_betas].round(4)
comparison['ecart B1 - A'] = (comparison['B1 melange'] - comparison['A exclusion']).round(4)
display(comparison)




classes d'age du tiers (variable centree) : [-18.92, -6.35, 13.15]


,A exclusion,B1 melange,ecart B1 - A
beta_age_2_I,-0.0260,-0.0261,-0.0001
beta_age_I,0.0339,0.0343,0.0004
beta_gender_2_female_I,-1.0663,-1.0692,-0.0029
beta_gender_female_I,1.0939,1.1208,0.0269
beta_maneuver_swerving_I,-0.4753,-0.5624,-0.0871
beta_maneuver_turning_left_I,-1.3106,-1.2629,0.0477
beta_point_of_impact_back_I_bike,-1.1169,-1.1491,-0.0322
beta_surface_condition_wet_I,-0.5648,-0.5550,0.0098
constant_I,0.8083,0.9247,0.1164


### Binary probit counterpart

With two outcomes the ordered logit *is* the binary logit estimated above, so
the ordered/unordered comparison has nothing to arbitrate here: only the link
function can be varied. `ordered_probit` over two discrete values is exactly a
binary probit. Its coefficients carry the opposite sign, because the ordered
parameterisation models P(severity = 1) = Phi(tau - V), and they are on the
normal rather than the logistic scale -- what is comparable is the
log-likelihood, hence the AIC.

The constant is dropped from the index: in the ordered parameterisation the
threshold `tau` plays that role, and keeping both would not be identified.


In [18]:
# --- MMV: binary probit counterpart of the binary logit above -----------------

index_mmv = (
      Beta('beta_gender_female_mmv_probit', 0, None, None, 0) * gender_female
    + Beta('beta_gender_2_female_mmv_probit', 0, None, None, 0)
      * (gender_2_female + gender_3_female)
    + Beta('beta_impact_back_bike_mmv_probit', 0, None, None, 0)
      * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
   # + Beta('beta_impact_back_epmd_mmv_probit', 0, None, None, 0)
   #   * point_of_impact_back * vehicle_e_pmd
    + Beta('beta_surface_wet_mmv_probit', 0, None, None, 0) * surface_condition_wet
    + Beta('beta_age_mmv_probit', 0, None, None, 0) * age
    + Beta('beta_swerving_mmv_probit', 0, None, None, 0) * maneuver_swerving
    + Beta('beta_turning_left_mmv_probit', 0, None, None, 0) * maneuver_turning_left
    + Beta('beta_age_opposite_mean_mmv_probit', 0, None, None, 0) * age_opposite_mean
  #  + Beta('beta_vehicle_2_e_pmd_mmv_probit', 0, None, None, 0)
  #    * (vehicle_2_e_pmd + vehicle_3_e_pmd)
   #       + beta_age_opposite_missing_I           * age_opposite_missing

)

model_name = 'binary_probit_mmv'
the_proba = ordered_probit(
    continuous_value=index_mmv,
    list_of_discrete_values=[1, 2],
    tau_parameter=Beta('tau_1_mmv_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, harmed))
model_mmv_probit = bio.BIOGEME(database_mmv_identified, logprob)
model_mmv_probit.modelName = model_name
results_mmv_probit = estimate_in('mmv', model_mmv_probit)

print(f'binary logit  : LL={results_logit_mmv.data.logLike:9.3f}  '
      f'K={results_logit_mmv.data.nparam:3d}  '
      f'AIC={results_logit_mmv.data.akaike:8.2f}')
print(f'binary probit : LL={results_mmv_probit.data.logLike:9.3f}  '
      f'K={results_mmv_probit.data.nparam:3d}  AIC={results_mmv_probit.data.akaike:8.2f}')
results_mmv_probit.get_estimated_parameters().round(4)


binary logit  : LL= -670.809  K=  9  AIC= 1359.62
binary probit : LL= -662.373  K=  9  AIC= 1342.75


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_mmv_probit,0.0201,0.0030,6.7315,0.0000
beta_age_opposite_mean_mmv_probit,-0.0151,0.0028,-5.3410,0.0000
beta_gender_2_female_mmv_probit,-0.6472,0.0860,-7.5252,0.0000
beta_gender_female_mmv_probit,0.6378,0.0969,6.5837,0.0000
beta_impact_back_bike_mmv_probit,-0.6718,0.1326,-5.0677,0.0000
beta_surface_wet_mmv_probit,-0.3295,0.1367,-2.4099,0.0160
beta_swerving_mmv_probit,-0.2850,0.1145,-2.4891,0.0128
beta_turning_left_mmv_probit,-0.7823,0.1914,-4.0868,0.0000
tau_1_mmv_probit,-0.4925,0.0582,-8.4653,0.0000


## Pedestrian

### Baseline (constants only)

Ordered-probit baseline with a flat utility (`continuous_value=0`) and the
single threshold `tau_1`.


In [19]:
model_name = 'ordered_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian_identified, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = estimate_in('pedestrian', model_cst_pedes)


### Full ordered-probit specification


In [20]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * (intersection_no_intersection)
    + beta_age_opposite_mean                             * age_opposite_mean
    + beta_age_opposite_missing                          * age_opposite_missing
    + beta_gender_2_female                   * gender_2_female
    + beta_user_category_pedestrian          * user_category_pedestrian
    # « Au moins un des tiers tourne » : la somme brute donnerait 2 si le
    # premier et le second tiers tournaient tous les deux, et ignorait le cas
    # ou seul le second tiers tourne.

    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [21]:
# --- Pedestrian: ordered-probit counterpart of the ordered logit above --------
# Same index (`utility_pedestrian`) and same threshold structure; only the link
# function changes. The two estimations are independent, so the Beta objects can
# be shared: each `estimate()` starts from the initial values again.

model_name = 'ordered_probit_pedestrian'

tau_1_pedes_probit = Beta('tau_1_pedes_probit', 1, None, None, 0)
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_pedes_probit,
)
logprob = log(Elem(the_proba, severity))
model_pedes_probit = bio.BIOGEME(database_pedestrian_identified, logprob)
model_pedes_probit.modelName = model_name
results_pedes_probit = estimate_in('pedestrian', model_pedes_probit)


results_pedes_probit.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0146,0.0016,9.0950,0.0000
beta_age_opposite_mean,-0.0110,0.0015,-7.3798,0.0000
beta_age_opposite_missing,2.1057,inf,0.0000,1.0000
beta_crossroad_traffic_lights,0.1978,0.0779,2.5384,0.0111
beta_gender_2_female,-0.5865,0.0653,-8.9887,0.0000
beta_gender_female,0.5110,0.0670,7.6228,0.0000
beta_intersection_no_intersection,0.2205,0.0694,3.1771,0.0015
beta_user_category_pedestrian,1.3204,0.0727,18.1737,0.0000
tau_1_pedes_probit,0.4903,0.0716,6.8497,0.0000
tau_1_pedes_probit_diff_2,4.1829,0.1666,25.1115,0.0000


### MNL counterpart of the same specification

The same nine variables, but one coefficient per alternative instead of a single
latent index. The ordered probit forces a variable to push severity in one
direction only; the MNL lets it raise the odds of injury and lower those of
death, at the cost of twice the coefficients.

The interaction `pedestrian x turning` never occurs among the fatalities, so its
coefficient on that alternative is not identified. The cell prints the counts.


In [22]:
# --- Pedestrian: binary logit, injury and fatality merged ---------------------
# Nine deaths in the segment cannot identify a fatality-specific utility, so the
# two harmful outcomes are merged and the model asks whether the person was
# harmed at all. The random intercept below is kept, commented out, as the mixed
# logit counterpart:
#
#     ASC_injury,n = asc_injury_ped + sigma_injury_ped * xi_n ,   xi_n ~ N(0, 1)
#
# It absorbs whatever raises or lowers the propensity to be injured and is not in
# the regressors. The choice probability is the logit probability integrated over
# `xi`, computed by Monte-Carlo simulation. `sigma_injury_ped` is the parameter
# that matters: not significantly different from zero means no unobserved
# heterogeneity to capture, and the MNL is enough.
#
# The same error component could be added to the fatality utility (a second
# `sigma`), at the cost of one more parameter and one more dimension to simulate.

from biogeme.expressions import MonteCarlo

xi_ped = bioDraws('xi_ped', 'NORMAL')
sigma_injury_ped = Beta('sigma_injury_ped', 1, None, None, 0)
beta_maneuver_2_turning = Beta('beta_maneuver_2_turning', 0, None, None, 0)
v_injury_ped_mnl = (Beta('asc_injury_ped', 0, None, None, 0)
   # + sigma_injury_ped * xi_ped
    + beta_gender_female_I                     * gender_female
    + beta_age_I                               * age

    + beta_intersection_no_intersection_I      * intersection_no_intersection
    + beta_age_opposite_mean_I                 * age_opposite_mean
    + beta_gender_2_female_I                   * gender_2_female
    + beta_user_category_pedestrian_I          * user_category_pedestrian
    + beta_crossroad_traffic_lights_I          * crossroad_traffic_lights
      
)

# LA FUSION SE FAIT SUR LA VARIABLE EXPLIQUEE, PAS SUR LES UTILITES.
#
# Ecrire `{1: 0, 2: V, 3: V}` ne fusionne pas la blessure et le deces : dans un
# logit la probabilite ne depend que de l'utilite, donc deux alternatives de
# meme utilite sont EQUIPROBABLES par construction -- la constante etant elle
# aussi partagee, rien ne rattrape le niveau. Le modele predisait donc autant de
# deces que de blesses, la ou le segment en compte 1 603 contre 9. Le cout
# etait de 1 117 points de log-vraisemblance a nombre de parametres identique
# (-2 175.35 contre -1 058.00), et surtout des coefficients estimes sous une
# contrainte que les donnees contredisent.
#
# On recode donc la reponse en deux modalites -- 1 indemne, 2 blesse ou tue --
# et on n'ecrit qu'une utilite. `harmed` est une expression Biogeme, ce qui
# evite de reconstruire la base de donnees.
utility_pedestrian_mnl = {1: 0, 2: v_injury_ped_mnl}

model_name = 'logit_pedestrian'

logprob = models.loglogit(utility_pedestrian_mnl, availability_harmed, harmed)

model_pedestrian_mnl = bio.BIOGEME(database_pedestrian_identified, logprob)
model_pedestrian_mnl.modelName = model_name
results_pedestrian_mnl = estimate_in('pedestrian', model_pedestrian_mnl)
results_pedestrian_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_injury_ped,-0.8037,0.1248,-6.4403,0.0000
beta_age_I,0.0265,0.0028,9.4060,0.0000
beta_age_opposite_mean_I,-0.0185,0.0026,-7.0040,0.0000
beta_crossroad_traffic_lights_I,0.3299,0.1495,2.2064,0.0274
beta_gender_2_female_I,-1.0954,0.1139,-9.6170,0.0000
beta_gender_female_I,0.9932,0.1163,8.5404,0.0000
beta_intersection_no_intersection_I,0.3507,0.1304,2.6897,0.0072
beta_user_category_pedestrian_I,2.2841,0.1237,18.4652,0.0000


In [23]:

model_name = 'logit_pedestrian_cst'

logprob = models.loglogit(
    {1: 0, 2: Beta('asc_harmed_ped_cst', 0, None, None, 0)},
    availability_harmed,
    harmed,
)
model_cst_pedes_mnl = bio.BIOGEME(database_pedestrian_identified, logprob)
model_cst_pedes_mnl.modelName = model_name
results_pedes_cst_mnl = estimate_in('pedestrian', model_cst_pedes_mnl)

print(f'LL(c) binary logit  : {results_pedes_cst_mnl.data.logLike:10.3f}  '
      f'(2 modalites de `harmed`)')
print(f'LL(c) ordered probit: {results_pedes_cst.data.logLike:10.3f}  '
      f'(3 niveaux de `severity`)')
results_pedes_cst_mnl.get_estimated_parameters().round(4)


LL(c) binary logit  :  -1900.612  (2 modalites de `harmed`)
LL(c) ordered probit:  -1956.268  (3 niveaux de `severity`)


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_harmed_ped_cst,0.3107,0.0383,8.1081,0.0


In [24]:
from biogeme.expressions import PanelLikelihoodTrajectory


In [25]:
database_pedestrian_panel_identified = db.Database(
    'database_pedestrian_panel_identified',
    df_pedestrian_identified.sort_values('Num_Acc').reset_index(drop=True))
database_pedestrian_panel_identified.panel('Num_Acc')

In [ ]:
# --- Pedestrian: panel likelihood, same specification as the MNL above --------
prob_ped_panel = models.logit(utility_pedestrian_mnl, availability_harmed, harmed)
logprob_ped_panel = log(PanelLikelihoodTrajectory(prob_ped_panel))

model_pedestrian_panel = bio.BIOGEME(database_pedestrian_panel_identified, logprob_ped_panel)
model_pedestrian_panel.modelName = 'logit_pedestrian_panel_identified'
results_pedestrian_panel = estimate_in('pedestrian', model_pedestrian_panel)

# --- Pedestrian: mixed logit, random term in the injury utility ---------------
utility_pedestrian_mixed = {
    1: utility_pedestrian_mnl[1],
    2: utility_pedestrian_mnl[2] + sigma_injury_ped * xi_ped,
}

prob_ped_mixed = models.logit(utility_pedestrian_mixed, availability_harmed, harmed)
logprob_ped_mixed = log(MonteCarlo(PanelLikelihoodTrajectory(prob_ped_mixed)))

model_pedestrian_mixed = bio.BIOGEME(database_pedestrian_panel_identified, logprob_ped_mixed,
                                      number_of_draws=1)
model_pedestrian_mixed.modelName = 'mixed_logit_pedestrian_panel_identified'
results_pedestrian_mixed = estimate_in('pedestrian', model_pedestrian_mixed)

print(f'panel MNL   : LL={results_pedestrian_panel.data.logLike:9.3f}  '
      f'K={results_pedestrian_panel.data.nparam:3d}  '
      f'AIC={results_pedestrian_panel.data.akaike:8.2f}')
print(f'mixed logit : LL={results_pedestrian_mixed.data.logLike:9.3f}  '
      f'K={results_pedestrian_mixed.data.nparam:3d}  '
      f'AIC={results_pedestrian_mixed.data.akaike:8.2f}')
print(results_pedestrian_panel.likelihood_ratio_test(results_pedestrian_mixed, 0.05))
results_pedestrian_mixed.get_estimated_parameters().round(4)


The number of draws (1) is low. The results may not be meaningful.


## Single-vehicle

### Baseline (constants only)

Ordered-probit baseline (`continuous_value=0`).


In [ ]:
model_name = 'ordered_probit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = estimate_in('single_vehicle', model_cst_solo)
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115844,0.064947,-32.578097,0.0
tau_1_diff_2,4.395907,0.099121,44.348801,0.0


### Full ordered-probit specification


In [ ]:
utility_sv = (
   beta_age * age +
    beta_user_category_passenger * user_category_passenger +
    beta_long_profile_slope *long_profile_slope 
     + beta_helmet_yes_ebike*helmet_yes*(vehicle_e_bike)
  #  + beta_number_of_passengers*user_category_driver*(number_of_passengers>0)
)


In [ ]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob)
model_solo_2.modelName = model_name
results_solo_2 = estimate_in('single_vehicle', model_solo_2)
results_solo_2.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.011141,0.003141,3.547091,3.895095e-04
beta_helmet_yes_ebike,-0.186226,0.128280,-1.451716,1.465807e-01
beta_long_profile_slope,0.303960,0.152742,1.990019,4.658889e-02
beta_user_category_passenger,-1.359845,0.227463,-5.978310,2.254648e-09
tau_1,-2.290106,0.086301,-26.536223,0.000000e+00
tau_1_diff_2,4.672434,0.123742,37.759480,0.000000e+00


### Ordered-logit counterpart

The same index with a logistic error term.


In [ ]:
# --- Single-vehicle: ordered-logit counterpart of the ordered probit above ----
# Same index (`utility_sv`), same thresholds; only the link function changes.

model_name = 'ordered_logit_sinv'

tau_1_solo_logit = Beta('tau_1_solo_logit', 1, None, None, 0)
the_proba = ordered_logit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_solo_logit,
)
logprob = log(Elem(the_proba, severity))
model_solo_logit = bio.BIOGEME(database_sv, logprob)
model_solo_logit.modelName = model_name
results_solo_logit = estimate_in('single_vehicle', model_solo_logit)

for label, results in (('ordered probit', results_solo_2),
                       ('ordered logit ', results_solo_logit)):
    print(f'{label} : LL={results.data.logLike:9.3f}  K={results.data.nparam:3d}  '
          f'AIC={results.data.akaike:8.2f}  BIC={results.data.bayesian:8.2f}')
results_solo_logit.get_estimated_parameters().round(4)

ordered probit : LL= -289.842  K=  6  AIC=  591.68  BIC=  625.89
ordered logit  : LL= -288.703  K=  6  AIC=  589.41  BIC=  623.62


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0272,0.0078,3.5041,0.0005
beta_helmet_yes_ebike,-0.4585,0.3971,-1.1545,0.2483
beta_long_profile_slope,0.7003,0.3694,1.8961,0.0580
beta_user_category_passenger,-2.9654,0.4072,-7.2831,0.0000
tau_1_solo_logit,-4.5354,0.2261,-20.0613,0.0000
tau_1_solo_logit_diff_2,9.2668,0.3186,29.0885,0.0000


### MNL counterpart of the same specification




In [ ]:
# --- Single-vehicle: MNL counterpart of the ordered probit above --------------
beta_helmet_yes_ebike_I = Beta('beta_helmet_yes_ebike_I', 0, None, None, 0)
beta_helmet_yes_ebike_F = Beta('beta_helmet_yes_ebike_F', 0, None, None, 0)

v_injury_sv = (Beta('asc_injury_sv', 0, None, None, 0)
    + beta_age_I * age
    + beta_user_category_passenger_I * user_category_passenger
    + beta_vehicle_e_pmd*vehicle_e_pmd

   # + beta_number_of_passengers_I * user_category_driver * number_of_passengers  # ajouté
)
v_fatality_sv = (Beta('asc_fatality_sv', 0, None, None, 0)
 +   beta_age_F * age +
    beta_long_profile_slope_F *long_profile_slope 
)


utility_sv_mnl = {1: 0, 2: v_injury_sv, 3: v_fatality_sv}

# Which coefficients cannot be identified on the fatality alternative.
print('Counts by severity (1 / 2 / 3) for the two interaction terms:')
for label, column in (('helmet_ebike', (df_sv['Helmet_Yes'] * df_sv['Vehicle_E-bike']) > 0),
                      ('passengers_driver',
                       (df_sv['User category_Driver'] * df_sv['Number of passengers']) > 0)):
    counts = df_sv.loc[column, 'severity'].value_counts().reindex([1, 2, 3], fill_value=0)
    print(f'  {label:18s} = 1 -> {counts[1]:4d} / {counts[2]:4d} / {counts[3]:4d}')
print()

model_name = 'mnl_sinv'
logprob = models.loglogit(utility_sv_mnl, availability, severity)
model_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_solo_mnl.modelName = model_name
results_solo_mnl = estimate_in('single_vehicle', model_solo_mnl)



Counts by severity (1 / 2 / 3) for the two interaction terms:
  helmet_ebike       = 1 ->    1 /   88 /    0
  passengers_driver  = 1 ->   19 /   46 /    0



In [ ]:
# --- Single-vehicle: constants-only counterpart of the MNL --------------------
# Three alternatives, as in `utility_sv_mnl`: one alternative-specific constant
# per harmful outcome, the reference being normalised to zero. With constants
# alone the model reproduces the observed severity shares exactly, so this
# LL(c) equals that of the ordered-probit baseline above -- the two are written
# separately only so that each model is referenced against its own framework.

model_name = 'mnl_sinv_cst'

logprob = models.loglogit(
    {1: 0,
     2: Beta('asc_injury_sv_cst', 0, None, None, 0),
     3: Beta('asc_fatality_sv_cst', 0, None, None, 0)},
    availability,
    severity,
)
model_cst_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_cst_solo_mnl.modelName = model_name
results_cst_solo_mnl = estimate_in('single_vehicle', model_cst_solo_mnl)

print(f'LL(c) MNL           : {results_cst_solo_mnl.data.logLike:10.3f}')
print(f'LL(c) ordered probit: {results_cst_solo.data.logLike:10.3f}')
results_cst_solo_mnl.get_estimated_parameters().round(4)


LL(c) MNL           :   -328.598
LL(c) ordered probit:   -328.598


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv_cst,-0.4189,0.2575,-1.6269,0.1038
asc_injury_sv_cst,4.0350,0.1636,24.6610,0.0000


In [ ]:
results_solo_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv,-0.5719,0.3214,-1.7792,0.0752
asc_injury_sv,4.8369,0.2450,19.7399,0.0000
beta_age_F,0.0705,0.0147,4.7845,0.0000
beta_age_I,0.0267,0.0127,2.0984,0.0359
beta_long_profile_slope_F,1.0120,0.4496,2.2508,0.0244
beta_user_category_passenger_I,-2.6405,0.3609,-7.3166,0.0000
beta_vehicle_e_pmd,-0.7133,0.2824,-2.5260,0.0115


## AIC and p(LR) of each model


In [ ]:

import math
from contextlib import contextmanager

from scipy.stats import chi2
from biogeme.expressions import Beta

EXACT_LR = True
AIC_THRESHOLD = math.sqrt(2)      # |t| au-dela duquel l'AIC garde le coefficient
SIGNIFICANCE_COLUMN = 'Keep (t-test 5%)'


def _parameter_columns(table):
    """Localise value / std err / t-test / p-value, versions robustes d'abord."""
    columns = list(table.columns)

    def find(keyword):
        matches = [c for c in columns if keyword in c.lower()]
        robust = [c for c in matches if 'rob' in c.lower()]
        return (robust or matches or [None])[0]

    value = next((c for c in columns if c.lower().strip() == 'value'), None)
    return {'value': value or columns[0], 'std': find('std err'),
            't': find('t-test'), 'p': find('p-value')}


# Les seuils d'un modele ordonne ne sont pas des variables : les fixer a zero ne
# revient pas a retirer un regresseur mais a casser la structure du modele
# (tau_1 = 0 deplace toute l'echelle, et annuler l'ecart tau_1_diff_2 confond les
# deux seuils, ce qui rend la vraisemblance degeneree -- d'ou les gradients NaN).
# Ils sont donc exclus de la selection.
THRESHOLD_PATTERN = re.compile(r'^(tau|delta)', re.IGNORECASE)





def _all_betas(expression):
    """Tous les objets Beta de l'arbre de l'expression."""
    found, stack, seen = [], [expression], set()
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        if isinstance(node, Beta):
            found.append(node)
        stack.extend(node.get_children())
    return found


@contextmanager
def _restricted_formula(expression, name, beta_values=None):

    betas = _all_betas(expression)
    targets = [beta for beta in betas if beta.name == name]
    if not targets:
        raise KeyError(f'{name} absent de la formule du modele')

    saved = [(beta, beta.status, beta.initValue) for beta in betas]
    if beta_values:
        for beta in betas:
            if beta.name in beta_values:
                beta.initValue = float(beta_values[beta.name])
    for beta in targets:
        beta.status = 1
        beta.initValue = 0.0
    try:
        yield
    finally:
        for beta, status, value in saved:
            beta.status = status
            beta.initValue = value


def lr_drop_test(the_biogeme, segment, results_full, parameter):
    """Reestime le modele avec `parameter` fixe a 0 ; retourne (LR, p, dAIC)."""
    formula = the_biogeme.log_like
    with _restricted_formula(formula, parameter, results_full.get_beta_values()):
        # L'objet Biogeme lit la formule a la construction : il doit etre
        # construit ET estime tant que le Beta est fixe.
        restricted = bio.BIOGEME(the_biogeme.database, formula,
                                 number_of_draws=the_biogeme.number_of_draws,
                                 generate_html=False, generate_pickle=False)
        restricted.modelName = f'{the_biogeme.modelName}_without_{parameter}'
        results_restricted = estimate_in(segment, restricted)

    statistic = 2 * (float(results_full.data.logLike)
                     - float(results_restricted.data.logLike))
    return (statistic, float(chi2.sf(statistic, 1)),
            float(results_restricted.data.akaike) - float(results_full.data.akaike))


def aic_selection_table(results, the_biogeme=None, segment=None, alpha=0.05):
    """Pour chaque coefficient : garde-t-on le terme sous AIC, sous t-test ?

    Si `the_biogeme` et `segment` sont fournis et `EXACT_LR` est vrai, les
    colonnes exactes (LR, p, Delta AIC) sont ajoutees, au prix d'une
    reestimation par coefficient.
    """
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)

    thresholds = [name for name in table.index if THRESHOLD_PATTERN.match(str(name))]
    if thresholds:
        print(f'  seuils exclus de la selection : {", ".join(thresholds)}')
        table = table.drop(index=thresholds)

    t_statistic = table[column['t']].astype(float)
    p_value = table[column['p']].astype(float)
    delta_aic = t_statistic ** 2 - 2

    selection = pd.DataFrame({
        'Value': table[column['value']].astype(float),
        't-test': t_statistic,
        'p-value': p_value,
        'Delta AIC (drop)': delta_aic,
        'Keep (AIC)': delta_aic > 0,
        SIGNIFICANCE_COLUMN: p_value < alpha,
    })

    if EXACT_LR and the_biogeme is not None and segment is not None:
        exact = {}
        for parameter in table.index:
            try:
                exact[parameter] = lr_drop_test(the_biogeme, segment,
                                                results, parameter)
            except Exception as error:
                print(f'  [{parameter}] LR test impossible : '
                      f'{type(error).__name__}: {error}')
                exact[parameter] = (float('nan'),) * 3
        exact = pd.DataFrame(exact, index=['LR', 'p (LR)',
                                           'Delta AIC (exact)']).T
        selection = selection.join(exact)
        selection['Keep (AIC, exact)'] = selection['Delta AIC (exact)'] > 0

    return selection.sort_values('Delta AIC (drop)', ascending=False)


# Le modele « principal » de chaque segment : celui que le tableau comparatif
# designe par l'AIC. (resultats, objet Biogeme, dossier de sortie).
MAIN_MODELS = {
    'Car crashes': (results_ml_motorized_vehicles, model_car, 'motorized_vehicles'),
    'MMV': (results_logit_mmv, model_mmv, 'mmv'),
    'Pedestrian': (results_pedestrian_mnl, model_pedestrian_mnl, 'pedestrian'),
    'Single-vehicle': (results_solo_mnl, model_solo_mnl, 'single_vehicle'),
}

aic_selection = {}
for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    print(f'=== {label} ===')
    table = aic_selection_table(results, the_biogeme, segment)
    aic_selection[label] = table

    keep_column = 'Keep (AIC, exact)' if 'Keep (AIC, exact)' in table else 'Keep (AIC)'
    kept = table.index[table[keep_column]].tolist()
    dropped = table.index[~table[keep_column]].tolist()
    only_aic = table.index[table[keep_column] & ~table[SIGNIFICANCE_COLUMN]].tolist()

    print(f'{len(kept)}/{len(table)} coefficients kept by AIC ({keep_column})')
    print(table.round(4))
    print(f'  dropped by AIC (|t| < {AIC_THRESHOLD:.3f}) : '
          f'{", ".join(dropped) if dropped else "none"}')
    print('  kept by AIC but not significant at 5% : '
          f'{", ".join(only_aic) if only_aic else "none"}')
    print()

=== Car crashes ===
16/16 coefficients kept by AIC (Keep (AIC, exact))
                                                     Value   t-test  p-value  \
constant_I                                          3.9195  35.0881   0.0000   
beta_vehicle_type_2_light_motorized_vehicle_I      -2.1153 -17.6603   0.0000   
beta_vehicle_type_2_large_motorized_vehicle_F       3.3328  12.1806   0.0000   
constant_F                                         -2.5447 -10.4053   0.0000   
beta_user_category_passenger_I                     -2.1494  -9.7281   0.0000   
beta_age_F                                          0.0544   5.8653   0.0000   
beta_number_of_involved_vehicles_I                 -0.9584  -4.6918   0.0000   
beta_gender_female_I                                0.6304   4.6681   0.0000   
beta_maneuver_2_turning_right_F                     1.0909   3.6428   0.0003   
beta_maneuver_without_change_of_direction_I         0.4106   3.5496   0.0004   
beta_age_I                                       

In [ ]:
# --- The four specifications --------------------------------------------------
# Definies ICI, en amont, parce que ce sont elles qui portent les LIBELLES : le
# tableau de selection AIC et celui des effets marginaux les reprennent, pour
# qu'une variable s'appelle pareil dans les trois tableaux du manuscrit.
# Les `results` sont des lambdas, donc rien n'est evalue a la definition : les
# modeles n'ont pas besoin d'exister quand cette cellule tourne.

MANUSCRIPT_SPECIFICATIONS = {
    'Car crashes': {
        'results': lambda: (first_defined('results_ml_motorized_vehicles'),
                            first_defined('results_constant_car')),
        'file': 'model_car.tex',
        'caption': ('Estimation results of the logistic regression model '
                    'predicting the injury severity of MMV riders in crashes '
                    'with motorized vehicles'),
        'label': 'tab:regression_results_car',
        'stats_caption': ('Model statistics of the logistic regression '
                          '(MMV vs. motorized vehicles)'),
        'stats_label': 'tab:regression_results_car_stats',
        'columns': ['Injury', 'Fatality'],
        'sections': [
            ('', [
                (r'Alternative specific constants',
                 {'Injury': 'constant_I', 'Fatality': 'constant_F'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('The individual is a female',
                 {'Injury': 'beta_gender_female_I'}),
                ('The individual is a passenger',
                 {'Injury': 'beta_user_category_passenger_I'}),
                ('Age of the individual (years)',
                 {'Injury': 'beta_age_I', 'Fatality': 'beta_age_F'}),
          
            ]),
            ('Collision characteristics', [
                ('Total number of vehicles and pedestrians involved in the crash',
                 {'Injury': 'beta_number_of_involved_vehicles_I'}),
                # Le modele n'a plus qu'un seul terme d'impact arriere, sans
                # distinction de vehicule (cellule 13).
                ('Rear impact on the MMV',
                 {'Injury': 'beta_point_of_impact_back_I'}),
                       ('The rider was not changing direction',
                 {'Injury': 'beta_maneuver_without_change_of_direction_I'}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred on a cycle facility',
                 {'Fatality': 'beta_accident_location_on_cycle_facility_F'}),
                # L'indicatrice est prise BRUTE (= 1 hors intersection) dans les
                # deux utilites : le libelle suit la variable estimee.
                ('Crash occurred away from an intersection',
                 {'Injury': 'beta_intersection_no_intersection_I',
                  'Fatality': 'beta_intersection_no_intersection_F'}),
                # `beta_vma_F` porte `vma * intersection_no_intersection` : la
                # vitesse n'entre QUE hors intersection, le terme est nul aux
                # intersections. Le libelle doit le dire, sinon le lecteur lit
                # un effet principal la ou il y a une interaction.
                ('Posted speed limit',
                 {'Injury': 'beta_vma_I',
                  'Fatality': 'beta_vma_F'}),
            ]),
            ('Environment characteristics', [
                # Les deux modalites de nuit sont regroupees en cellule 13 :
                # un seul coefficient, porte par le parametre `..._on_F`.
                ('Crash occurred at night (with or without street lighting)',
                 {'Fatality':
                  'beta_lighting_conditions_night_with_street_lightings_on_F'}),
            ]),
            ('Second-party characteristics', [
                ('A second-party vehicle is a light vehicle',
                 {'Injury': 'beta_vehicle_type_2_light_motorized_vehicle_I',
                  'Fatality': 'beta_vehicle_type_2_light_motorized_vehicle_F'}),
                ('A second-party vehicle is a heavy vehicle',
                 {'Fatality': 'beta_vehicle_type_2_large_motorized_vehicle_F'}),
                ('A second-party vehicle is turning right',
                 {'Fatality': 'beta_maneuver_2_turning_right_F'}),
                ('A second-party vehicle is overtaking',
                 {'Injury': 'beta_maneuver_2_overtaking_I'}),
            ]),
        ],
    },
    'MMV': {
        'full_sample': lambda: len(df_mmv),
        'dropped_note': ('observations in which the opposing rider left the scene are excluded since their age and sex are not recorded'),
        'results': lambda: (first_defined('results_logit_mmv'),
                            first_defined('results_constant_mmv')),
        'file': 'model_mmv.tex',
        'caption': ('Estimation results of the binary logistic regression '
                    'predicting the injury severity of MMV riders in crashes '
                    'with other MMVs'),
        'label': 'tab:regression_results_mmv',
        'stats_caption': ('Model statistics of the binary logistic regression '
                          'predicting the injury severity of MMV riders in '
                          'crashes with other MMVs'),
        'stats_label': 'tab:regression_stats_mmv',
        'columns': ['Injury'],
        'sections': [
            ('', [
                (r'Alternative specific constant',
                 {'Injury': 'constant_I'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('Individual is a female', {'Injury': 'beta_gender_female_I'}),
                ('Age of the individual (years)', {'Injury': 'beta_age_I'}),
            ]),
            ('Collision characteristics', [
                ('The individual was swerving',
                 {'Injury': 'beta_maneuver_swerving_I'}),
                ('The individual was turning left',
                 {'Injury': 'beta_maneuver_turning_left_I'}),
                ('Rear impact on the e-PMD',
                 {'Injury': 'beta_point_of_impact_back_I_epmd'}),
                ('Rear impact on the (e-)bike',
                 {'Injury': 'beta_point_of_impact_back_I_bike'}),
                ('Rear impact on the MMV',
                 {'Injury': 'beta_point_of_impact_back_I'}),
            ]),
            ('Environment characteristics', [
                ('Wet surface', {'Injury': 'beta_surface_condition_wet_I'}),
            ]),
            ('Second-party characteristics', [
                ('At least one second-party individual is a female',
                 {'Injury': 'beta_gender_2_female_I'}),
                ('Mean age of the second-party individuals (years)',
                 {'Injury': 'beta_age_2_I'}),
                ('Second party not identified',
                 {'Injury': 'beta_age_opposite_missing_I'}),
                ('At least one second-party is an e-PMD',
                 {'Injury': 'beta_vehicle_2_e_pmd_I'}),
            ]),
        ],
    },


    # --- Les contreparties MNL ------------------------------------------------
    # Memes segments, memes donnees, mais deux utilites au lieu d'un score
    # ordonne : chaque variable recoit un coefficient propre a la blessure et un
    # autre au deces, d'ou deux colonnes. Une variable absente d'une des deux
    # utilites imprime un tiret -- c'est le cas pour la moitie du modele sans
    # tiers, ou 25 deces ne permettent pas d'identifier plus de deux
    # coefficients sur l'alternative mortelle.
    'Pedestrian (MNL)': {
        'full_sample': lambda: len(df_pedestrian),
            'dropped_note': ('observations in which the opposing rider left the scene are excluded since their age and sex are not recorded'),
        # LL(c) du logit binaire sur `harmed`, pas celui du probit ordonne a
        # trois niveaux : ce n'est pas la meme variable expliquee.
        'results': lambda: (first_defined('results_pedestrian_mnl',
                                          'results_pedes_mnl'),
                            first_defined('results_pedes_cst_mnl',
                                          'results_pedes_cst')),
        'file': 'model_ped_mnl.tex',
        'caption': ('Estimation results of the logistic regression for the injury severity of '
                    'MMV riders and pedestrians in crashes between MMVs and '
                    'pedestrians'),
        'label': 'tab:regression_results_pedes_mnl',
        'stats_caption': ('Model statistics of the logistic regression predicting the injury '
                          'severity of MMV riders and pedestrians'),
        'stats_label': 'tab:regression_results_pedes_mnl_stats',
        # UNE seule colonne : l'utilite de la blessure et celle du deces sont la
        # meme expression dans la cellule d'estimation, donc il n'existe qu'un
        # jeu de coefficients, suffixes `_I`.
        'columns': ['Estimates'],
        'sections': [
            ('', [
                ('Alternative specific constant', {'Estimates': 'asc_injury_ped'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('The individual is a female',
                 {'Estimates': 'beta_gender_female_I'}),
                ('The individual is a pedestrian',
                 {'Estimates': 'beta_user_category_pedestrian_I'}),
                ('Age of the individual (years)', {'Estimates': 'beta_age_I'}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred away from an intersection',
                 {'Estimates': 'beta_intersection_no_intersection_I'}),
                ('Crash occurred at a signalized intersection',
                 {'Estimates': 'beta_crossroad_traffic_lights_I'}),
            ]),
            ('Second-party characteristics', [
                ('At least one second-party individual is a female',
                 {'Estimates': 'beta_gender_2_female_I'}),
                ('Mean age of the second-party individuals (years)',
                 {'Estimates': 'beta_age_opposite_mean_I'}),
                ('Second party not identified',
                 {'Estimates': 'beta_age_opposite_missing_I'}),
                       ('The pedestrian was hit by a turning second-party vehicle',
                 {'Estimates': 'beta_maneuver_2_turning'}),
            ]),
      
        ],
    },
    'Single-vehicle (MNL)': {
        'results': lambda: (first_defined('results_solo_mnl'),
                            first_defined('results_cst_solo_mnl',
                                          'results_cst_solo')),
        'file': 'model_solo_mnl.tex',
        'caption': ('Estimation results of the multinomial logistic regression for the injury '
                    'severity of MMV riders in single-vehicle crashes'),
        'label': 'tab:solo_crashes_mnl',
        'stats_caption': ('Model statistics of the multinomial logistic regression for the '
                          'injury severity of MMV riders in single-vehicle '
                          'crashes'),
        'stats_label': 'tab:solo_crashes_mnl_stats',
        'columns': ['Injury', 'Fatality'],
        'sections': [
            ('', [
                ('Alternative specific constants',
                 {'Injury': 'asc_injury_sv', 'Fatality': 'asc_fatality_sv'}),
            ]),
            ('Individual and vehicle characteristics', [
                ('Age of the individual (years)',
                 {'Injury': 'beta_age_I', 'Fatality': 'beta_age_F'}),
                ('The individual is a passenger',
                 {'Injury': 'beta_user_category_passenger_I',
                  'Fatality': None}),
                ('The rider was on an e-PMD',
                 {'Injury': 'beta_vehicle_e_pmd', 'Fatality': None}),
            ]),
            ('Infrastructure characteristics', [
                ('Crash occurred on a slope',
                 {'Injury': None, 'Fatality': 'beta_long_profile_slope_F'}),
            ]),

        ],
   
    },
}


# --- Un seul libelle par variable, pour les trois tableaux --------------------
# `MANUSCRIPT_SPECIFICATIONS` est la source unique. On en tire deux index :
#   - par nom de Beta, pour le tableau de selection AIC, dont les lignes SONT
#     des parametres ;
#   - par nom de colonne, pour les effets marginaux, dont les lignes sont des
#     variables du jeu de donnees.
# Le second se deduit du premier : un Beta s'appelle `beta_<colonne normalisee>`
# suivi de l'alternative, donc on retire le prefixe et le suffixe.

def _manuscript_label_index():
    """{nom de Beta: libelle} et {colonne normalisee: libelle}."""
    by_parameter, by_column = {}, {}
    for specification in MANUSCRIPT_SPECIFICATIONS.values():
        for _, rows in specification['sections']:
            for label, parameters in rows:
                for name in parameters.values():
                    if not name:
                        continue
                    by_parameter[name] = label
                    slug = re.sub(r'^(beta|asc)_', '', str(name))
                    slug = re.sub(r'_(I|F)$', '', slug)
                    by_column.setdefault(slug, label)
    return by_parameter, by_column


MANUSCRIPT_PARAMETER_LABELS, MANUSCRIPT_COLUMN_LABELS = _manuscript_label_index()
print(f'{len(MANUSCRIPT_PARAMETER_LABELS)} libelles de parametres, '
      f'{len(MANUSCRIPT_COLUMN_LABELS)} de colonnes')


38 libelles de parametres, 33 de colonnes


In [ ]:
# --- LaTeX ------------------------------------------------


from latex_tables import colorize

import re

AIC_TABLES_DIRECTORY = RESULTS_ROOT / 'tables'

# Surcharges de libelles, ex. {'beta_age_I': 'Age (years)'}.
# Constantes alternatives-specifiques et seuils des modeles ordonnes. Defini
# ICI parce que c'est la premiere cellule qui en a besoin ; la cellule de
# comparaison des cadres, plus bas, reutilise les memes fonctions.
INTERCEPT_PARAMETER = re.compile(r'^(asc|constant|tau)', re.IGNORECASE)


def _count_intercept_names(names):
    """Constantes alternatives-specifiques et seuils parmi des noms."""
    return sum(1 for name in names if INTERCEPT_PARAMETER.match(str(name)))


def _count_intercepts(results):
    """Idem, a partir d'un objet de resultats Biogeme."""
    try:
        return _count_intercept_names(results.get_estimated_parameters().index)
    except Exception:                       # noqa: BLE001
        return 0


PARAMETER_LABELS = {}

ALTERNATIVE_LABELS = {'_I': 'injury', '_F': 'fatality'}


def _escape(text):
    for char in ('&', '%', '$', '#', '_', '{', '}'):
        text = text.replace(char, '\\' + char)
    return text


def _row(cells):
    return ' & '.join(cells) + r' \\'


def _pretty(name):
    """`beta_point_of_impact_back_F` -> `Point of impact back (fatality)`.

    Le libelle des tableaux d'estimation du manuscrit est prioritaire : une
    variable doit s'appeler pareil dans le tableau du modele, dans celui de la
    selection AIC et dans celui des effets marginaux. `PARAMETER_LABELS` reste
    disponible pour les parametres qui ne figurent dans aucune specification.
    """
    if name in PARAMETER_LABELS:
        return PARAMETER_LABELS[name]
    manuscript = MANUSCRIPT_PARAMETER_LABELS.get(name)
    if manuscript:
        alternative = next((text for suffix, text in ALTERNATIVE_LABELS.items()
                            if str(name).endswith(suffix)), None)
        return f'{manuscript} ({alternative})' if alternative else manuscript
    label = str(name)
    alternative = next((text for suffix, text in ALTERNATIVE_LABELS.items()
                        if label.endswith(suffix)), None)
    label = re.sub(r'^(beta|asc)[_ ]', '', label)
    label = re.sub(r'[_ ](I|F)$', '', label)
    label = label.replace('_', ' ').strip()
    label = label[:1].upper() + label[1:]
    return f'{label} ({alternative})' if alternative else label


def _fmt_p(value):
    if value != value:                      # NaN
        return ''
    return '$<$0.001' if float(value) < 0.001 else f'{float(value):.3f}'


def make_aic_selection_table(selection, caption=None, label=None, alpha=0.05,
                            drop_intercepts=True):
    """Estimation, t, cout en AIC du retrait du coefficient, p-value du LR.

    La colonne `Delta AIC (exact)` n'existe que si la selection a ete calculee
    avec re-estimation ; sinon on retombe sur `Delta AIC (drop)`, obtenu en
    fixant le coefficient a zero, et l'en-tete porte un obele pour le dire.
    """
    # Les constantes sont ECARTEES du tableau. Retirer un intercept n'est pas
    # retirer une variable : le test dirait seulement que la constante n'est pas
    # nulle, ce qui n'est pas une question de specification, et son Delta AIC
    # ecrase l'echelle de la colonne -- 1 882 points pour la constante du modele
    # vehicule-seul contre 42 pour la premiere vraie variable.
    if drop_intercepts:
        selection = selection.loc[
            [not INTERCEPT_PARAMETER.match(str(name)) for name in selection.index]]

    exact = 'p (LR)' in selection.columns
    delta_column = ('Delta AIC (exact)' if 'Delta AIC (exact)' in selection.columns
                    else 'Delta AIC (drop)')
    delta_header = (r'$\Delta$AIC' if delta_column.endswith('(exact)')
                    else r'$\Delta$AIC$^{\dagger}$')

    columns = ['Parameter', 'Estimate', '$t$', delta_header]
    if exact:
        columns.append('$p$ (LR)')
    alignment = 'l' + ' r' * (len(columns) - 1)

    lines = [
        r'\begin{table}[H]',
        r'\centering',
        r'\small',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        _row(columns),
        r'\hline',
    ]

    for parameter, record in selection.iterrows():
        delta = record[delta_column] if delta_column in record else float('nan')
        cells = [_escape(_pretty(parameter)),
                 f"{float(record['Value']):.3f}",
                 f"{float(record['t-test']):.2f}",
                 '' if delta != delta else f'{float(delta):.2f}']
        if exact:
            cells.append(_fmt_p(record['p (LR)']))
        lines.append(_row(cells))

    # Rien en note de bas de tableau : les colonnes calculees sont expliquees
    # dans la legende, ou le lecteur les rencontre AVANT de lire les chiffres
    # plutot qu'apres.
    lines.append(r'\hline')
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(line for line in lines if line)


AIC_TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)


SEGMENT_DESCRIPTIONS = {
    'Car crashes': 'crashes between MMVs and motorized vehicles',
    'MMV': 'crashes between MMVs and other MMVs',
    'Pedestrian': 'crashes between MMVs and pedestrians',
    'Single-vehicle': 'single-MMV crashes',
}

aic_selection_tex = {}
for label, selection in aic_selection.items():
    stem = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    described = SEGMENT_DESCRIPTIONS.get(label, label)
    tex = make_aic_selection_table(
        selection,
        caption=('Contribution of each estimated coefficient to the goodness '
                 f'of fit of the model of {described}. '
                 r'$\Delta$AIC is AIC(without) $-$ AIC(with): a positive value '
                 'means that the model fits better with the coefficient than '
                 'without it. '
                 r'$p$ (LR) is the p-value of a likelihood ratio test between '
                 'the model and the same model re-estimated without that '
                 'single coefficient.'),
        label=f'tab:aic_selection_{stem}',
    )
    (AIC_TABLES_DIRECTORY / f'aic_selection_{stem}.tex').write_text(
        colorize(tex) + '\n', encoding='utf-8')
    aic_selection_tex[label] = tex
    print(f'{AIC_TABLES_DIRECTORY / f"aic_selection_{stem}.tex"}')

print()
print(next(iter(aic_selection_tex.values())))

results/tables/aic_selection_car_crashes.tex
results/tables/aic_selection_mmv.tex
results/tables/aic_selection_pedestrian.tex
results/tables/aic_selection_single_vehicle.tex

\begin{table}[H]
\centering
\small
\caption{Contribution of each estimated coefficient to the goodness of fit of the model of crashes between MMVs and motorized vehicles. $\Delta$AIC is AIC(without) $-$ AIC(with): a positive value means that the model fits better with the coefficient than without it. $p$ (LR) is the p-value of a likelihood ratio test between the model and the same model re-estimated without that single coefficient.}
\label{tab:aic_selection_car_crashes}
\begin{tabular}{l r r r r}
\hline
Parameter & Estimate & $t$ & $\Delta$AIC & $p$ (LR) \\
\hline
A second-party vehicle is a light vehicle (injury) & -2.115 & -17.66 & 306.11 & $<$0.001 \\
A second-party vehicle is a heavy vehicle (fatality) & 3.333 & 12.18 & 107.93 & $<$0.001 \\
The individual is a passenger (injury) & -2.149 & -9.73 & 90.10 & $<$0

## AME


In [ ]:

from biogeme.expressions import Derive, Variable, exp

# Au-dela de ce nombre de valeurs distinctes, la variable est traitee comme
# continue (derivee) ; en deca, comme un comptage (variation discrete de +1).
DISCRETE_MAX_LEVELS = 10

BASELINE_LABEL = 'Baseline predicted probabilities'



DUMMY_PREFIXES = [
    'Crossroad', 'Helmet_driver', 'Helmet', 'Point of impact_2',
    'Point of impact_3', 'Point of impact', 'Gender_driver', 'Gender_2',
    'Gender_3', 'Gender', 'vehicle_type_2', 'vehicle_type_3', 'Vehicle_2',
    'Vehicle_3', 'Vehicle', 'Pavement', 'Intersection', 'User category',
    'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration',
    'Age category', 'Accident location', 'Surface condition', 'Maneuver_2',
    'Maneuver_3', 'Maneuver', 'Pedestrian localisation', 'Pedestrian action',
    'Max speed', 'Long profile', 'Weather conditions', 'Road type',
    'Trip purpose', 'Reflective jacket', 'plan', 'Obstacle', 'age_driver',
    'Year',
]

# Libelles imposes : continues, variables construites, ou tournures plus claires.
VARIABLE_LABELS = {
    'age': 'Age of the individual (years)',
    'age_2': "Age of the third party's driver (years)",
    'age_opposite_mean': "Age of the second-party's driver (years)",
    'age_opposite_missing': 'Second party not identified',
    'vma': 'Posted speed limit (km/h)',
    'number of involved vehicles': 'Number of vehicles involved',
    'Number of passengers': 'Number of passengers',
    'long_profile_slope': 'Slope',
    'Long profile_Slope': 'Slope',
    'Intersection_No intersection': 'Away from an intersection',
    'User category_Passenger': 'Passenger (vs. driver)',
    'User category_Pedestrian': 'Pedestrian',
    'Crossroad_Traffic lights': 'Signalized crossroad',
    'Accident location_On cycle facility': 'Presence of a cycle facility',
}


def _pretty_variable(name):
    """Nom lisible d'une colonne du modele.

    L'ordre compte. Le libelle du tableau d'estimation du manuscrit PASSE EN
    PREMIER : une variable doit porter le meme nom dans le tableau du modele,
    dans celui de la selection AIC et ici. `VARIABLE_LABELS` ne sert donc plus
    qu'aux colonnes absentes des specifications, et le decoupage par prefixe,
    qu'a celles qu'aucun des deux ne nomme.

    Les libelles de GROUPE de `KEY_VARIABLES` (« Second-party heavy vehicle »,
    qui couvre deux colonnes) traversent sans etre reecrits : ce ne sont pas des
    noms de colonnes, ils ne correspondent a aucune entree des index.
    """
    manuscript = MANUSCRIPT_COLUMN_LABELS.get(normalize(str(name)))
    if manuscript:
        return manuscript
    if name in VARIABLE_LABELS:
        return VARIABLE_LABELS[name]
    for prefix in DUMMY_PREFIXES:
        if name.startswith(f'{prefix}_'):
            category = name[len(prefix) + 1:].replace('_', ' ')
            variable = prefix.replace('_', ' ').replace('vehicle type 2',
                                                        'Third-party vehicle')
            variable = variable[:1].upper() + variable[1:]
            return f'{variable}: {category.lower()}'
    label = name.replace('_', ' ').strip()
    return label[:1].upper() + label[1:]


SEGMENT_DISPLAY_NAMES = {
    'Car crashes': 'Motorized vehicles',
}


def display_name(segment):
    """Libelle affiche d'un segment."""
    return SEGMENT_DISPLAY_NAMES.get(segment, segment)


KEY_VARIABLES = {
    'Car crashes': [
        'age', 'number of involved vehicles',
        ('Second-party heavy vehicle',
         ['vehicle_type_2_Large motorized vehicle',
          'vehicle_type_3_Large motorized vehicle']),
        ('Second-party light vehicle',
         ['vehicle_type_2_Light motorized vehicle',
          'vehicle_type_3_Light motorized vehicle']),
        ('Crash at night',
         ['Lighting conditions_Night with street lightings on',
          'Lighting conditions_Night without street lightings']),
        'Accident location_On cycle facility',
        'Intersection_No intersection', 'vma',
        ('Second-party turning right',
         ['Maneuver_2_Turning right', 'Maneuver_3_Turning right']),
        'User category_Passenger', 'Gender_Female',
    ],
    'MMV': [
        'age', 'age_opposite_mean', 'Gender_Female', 'Surface condition_Wet',
        'Maneuver_Swerving', 'Maneuver_Turning left', 'Point of impact_Back',
        'gender_2_female'
    ],
    'Pedestrian': [
        'age', 'age_opposite_mean', 'Gender_Female', 'User category_Pedestrian',
        'Crossroad_Traffic lights', 'Intersection_No intersection', 'gender_2_female'
    ],
    'Single-vehicle': [
        'age', 'User category_Passenger', 'Long profile_Slope',
        'Number of passengers',
    ],
}


def _model_variables(the_biogeme, exclude=('severity',)):
    """Noms des colonnes qui apparaissent dans la formule du modele."""
    found, stack, seen = set(), [the_biogeme.log_like], set()
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        if isinstance(node, Variable):
            found.add(node.name)
        stack.extend(node.get_children())
    return sorted(found - set(exclude))


def _outcome_simulators(the_biogeme, data, outcomes, choice_column='severity'):
    """Un objet Biogeme par modalite, pretes a simuler P(severity = j)."""
    simulators = {}
    for outcome in outcomes:
        frame = data.copy()
        frame[choice_column] = outcome
        simulator = bio.BIOGEME(db.Database('marginal_effects', frame),
                                {'probability': exp(the_biogeme.log_like)},
                                number_of_draws=the_biogeme.number_of_draws,
                                generate_html=False, generate_pickle=False)
        simulator.modelName = f'{the_biogeme.modelName}_simulation'
        simulators[outcome] = simulator
    return simulators


def _derivative_simulators(the_biogeme, data, outcomes, column,
                           choice_column='severity'):
    """Un objet Biogeme par modalite, pretes a simuler dP(severity = j)/dx.

    `Derive` derive l'expression symboliquement : pas de pas a choisir, pas de
    perte de precision par soustraction de deux nombres proches, et la regle de
    derivation en chaine traverse tous les termes ou `column` apparait.
    """
    simulators = {}
    for outcome in outcomes:
        frame = data.copy()
        frame[choice_column] = outcome
        simulator = bio.BIOGEME(
            db.Database('marginal_effects', frame),
            {'derivative': Derive(exp(the_biogeme.log_like), column)},
            number_of_draws=the_biogeme.number_of_draws,
            generate_html=False, generate_pickle=False)
        simulator.modelName = f'{the_biogeme.modelName}_derivative'
        simulators[outcome] = simulator
    return simulators


def marginal_effects(the_biogeme, results, variables=None, outcome_labels=None,
                     choice_column='severity'):
    """Effets marginaux moyens, en points de pourcentage.

    Une ligne par variable, une colonne par modalite. `variables` limite le
    tableau aux variables voulues ; None prend toutes celles du modele.
    """
    data = the_biogeme.database.data.copy()
    betas = results.get_beta_values()
    outcomes = sorted(int(value) for value in data[choice_column].unique())
    labels = outcome_labels or {outcome: f'P(severity = {outcome})'
                                for outcome in outcomes}

    available = _model_variables(the_biogeme, exclude=(choice_column,))

    def normalise(entry):
        """'colonne' ou (libelle, [colonnes]) -> (libelle, colonnes retenues)."""
        if isinstance(entry, str):
            return entry, [entry]
        label, columns = entry
        return label, list(columns)

    # `_model_variables` rend les noms de COLONNES du jeu de donnees
    # (`Gender_2_Female`), parce que les `Variable` de la formule ont ete creees
    # avec le libelle d'origine. Une entree de `KEY_VARIABLES` est parfois
    # ecrite a la mode des Beta (`gender_2_female`), qui est ce meme nom
    # normalise. On resout donc par le nom normalise, faute de quoi la variable
    # serait declaree « hors du modele » alors qu'elle y est.
    available_by_slug = {normalize(name): name for name in available}

    def resolve(column):
        """Nom de colonne reel correspondant a `column`, ou None."""
        if column in available:
            return column
        return available_by_slug.get(normalize(str(column)))

    if variables is None:
        selection = [(name, [name]) for name in available]
    else:
        selection = []
        for entry in variables:
            label, columns = normalise(entry)
            resolved = [(column, resolve(column)) for column in columns]
            kept = [found for _, found in resolved if found]
            missing = [column for column, found in resolved if not found]
            if missing:
                print(f'  [{label}] hors du modele, ignoree : {", ".join(missing)}')
            if kept:
                selection.append((label, kept))

    records = []
    for label, columns in selection:
        derivative_column = None
        column = data[columns[0]]
        values = set(pd.concat([data[name] for name in columns]).dropna().unique())
        if values <= {0, 1}:
            # Groupe d'indicatrices : « aucune » contre « la premiere ». Mettre
            # toutes les colonnes a 1 donnerait 2 sur un terme somme, et n'a pas
            # de sens pour des modalites qui s'excluent (nuit avec / sans
            # eclairage).
            scale, kind = 1.0, 'dummy (0 to 1)'
            frame_low = data.assign(**{name: 0.0 for name in columns})
            frame_high = data.assign(**{name: 1.0 if name == columns[0] else 0.0
                                        for name in columns})
        elif column.nunique() <= DISCRETE_MAX_LEVELS:
            # Comptage : une unite de plus, pas une derivee locale extrapolee.
            scale, kind = 1.0, 'per unit (+1)'
            frame_low = data
            frame_high = data.assign(**{columns[0]: column + 1})
        else:
            if len(columns) > 1:
                print(f'  [{label}] groupe continu non gere, ignoree')
                continue
            if column.nunique() < 2:
                continue                       # variable constante : pas d'effet
            # Derivee exacte de la probabilite, moyennee sur l'echantillon.
            scale, kind, derivative_column = 1.0, 'per unit', columns[0]

        if derivative_column is not None:
            simulators = _derivative_simulators(the_biogeme, data, outcomes,
                                                derivative_column, choice_column)

            def effect(outcome):
                simulated = simulators[outcome].simulate(betas)['derivative']
                return 100 * float(simulated.mean())
        else:
            low = _outcome_simulators(the_biogeme, frame_low, outcomes,
                                      choice_column)
            high = _outcome_simulators(the_biogeme, frame_high, outcomes,
                                       choice_column)

            def effect(outcome):
                difference = (high[outcome].simulate(betas)['probability']
                              - low[outcome].simulate(betas)['probability'])
                return 100 * float(difference.mean()) / scale

        record = {'Variable': _pretty_variable(label), 'Variation': kind}
        for outcome in outcomes:
            record[labels[outcome]] = effect(outcome)
        records.append(record)

    table = pd.DataFrame.from_records(records).set_index('Variable')
    point_columns = [labels[outcome] for outcome in outcomes]
    ordering = table[point_columns].abs().max(axis=1)
    table = table.loc[ordering.sort_values(ascending=False).index]


    simulators = _outcome_simulators(the_biogeme, data, outcomes, choice_column)
    baseline = {'Variable': BASELINE_LABEL, 'Variation': 'baseline'}
    for outcome in outcomes:
        baseline[labels[outcome]] = 100 * float(
            simulators[outcome].simulate(betas)['probability'].mean())
    baseline = pd.DataFrame.from_records([baseline]).set_index('Variable')
    return pd.concat([baseline, table])


SEVERITY_LABELS = {1: 'No injury', 2: 'Injury', 3: 'Fatality'}

marginal_effects_tables = {}
for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    print(f'=== {label} : average marginal effects (percentage points) ===')
    try:
        table = marginal_effects(the_biogeme, results,
                                 variables=KEY_VARIABLES.get(label),
                                 outcome_labels=SEVERITY_LABELS)
    except Exception as error:
        print(f'  calcul impossible : {type(error).__name__}: {error}\n')
        continue
    marginal_effects_tables[label] = table
    print(table.round(2).to_string())
    print()


# --- Same tables, in LaTeX ----------------------------------------------------
def make_marginal_effects_table(table, caption=None, label=None):
    """Tableau LaTeX des effets marginaux moyens (points de pourcentage)."""
    outcomes = [column for column in table.columns if column != 'Variation']
    columns = ['Variable'] + outcomes
    alignment = 'l' + ' c' * len(outcomes)

    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        _row([_escape(name) for name in columns]),
        r'\hline',
    ]
    for variable, record in table.iterrows():
        is_baseline = record.get('Variation') == 'baseline'
        cells = [r'\quad \textit{' + _escape(str(variable)) + '}' if is_baseline
                 else _escape(str(variable))]
        for outcome in outcomes:
            cells.append(f'{float(record[outcome]):.2f}')
        lines.append(_row(cells))
        if is_baseline:
            lines.append(r'\hline')
    lines += [r'\hline', r'\end{tabular}', r'\medskip',
              r'\footnotesize Average marginal effects, in percentage points'
              r'. For a dummy, the effect of '
              r'moving it from 0 to 1; for a count, the effect of one more unit '
              r'($x \\rightarrow x+1$); for a continuous variable, the '
              r'derivative. Averaged over the estimation sample. The first '
              r'row gives the mean predicted probability of each outcome, '
              r'against which the effects are to be read.',
              r'\end{table}']
    return '\n'.join(line for line in lines if line)


marginal_effects_tex = {}
for label, table in marginal_effects_tables.items():
    stem = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
    tex = make_marginal_effects_table(
        table,
        caption=(f'{display_name(label)}: average marginal effects '
                 f'(percentage points).'),
        label=f'tab:marginal_effects_{stem}',
    )
    (AIC_TABLES_DIRECTORY / f'marginal_effects_{stem}.tex').write_text(colorize(tex),
                                                                      encoding='utf-8')
    marginal_effects_tex[label] = tex
    print(AIC_TABLES_DIRECTORY / f'marginal_effects_{stem}.tex')



def make_combined_marginal_effects_table(
    tables, outcomes=None,
    caption=('Average marginal effects (percentage points) of the key variables, '
             'by segment.'),
    label='tab:marginal_effects_all',
):
    outcomes = outcomes or list(SEVERITY_LABELS.values())
    columns = ['Variable'] + outcomes
    width = len(columns)

    lines = [
        r'\begin{table}[H]', r'\centering', r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{l{" c" * len(outcomes)}}}',
        r'\hline',
        _row([_escape(name) for name in columns]),
        r'\hline',
    ]

    def _formatted(record, outcome, is_baseline):
        if outcome not in record:
            return None
        return f'{float(record[outcome]):.2f}'

    def _merges_harmful_outcomes(table):

        if len(outcomes) < 3:
            return False
        injury, fatality = outcomes[1], outcomes[2]
        if injury not in table.columns or fatality not in table.columns:
            return False
        return bool((table[injury] == table[fatality]).all())

    for position, (segment, table) in enumerate(tables.items()):
        if position:
            lines.append(r'\hline')
        lines.append(rf'\multicolumn{{{width}}}{{l}}'
                     rf'{{\textit{{{_escape(display_name(segment))}}}}} \\')
        merged = _merges_harmful_outcomes(table)
        for variable, record in table.iterrows():
            is_baseline = record.get('Variation') == 'baseline'
            cells = [r'\quad \textit{' + _escape(str(variable)) + '}' if is_baseline
                     else _escape(str(variable))]
            for outcome in outcomes:
                value = _formatted(record, outcome, is_baseline)
                if merged and outcome == outcomes[1]:
                    # Une seule valeur, centree sur les deux colonnes.
                    cells.append(r'\multicolumn{2}{c}{' + (value or '') + '}')
                    continue
                if merged and outcome == outcomes[2]:
                    continue
                cells.append('' if value is None else value)
            lines.append(_row(cells))

    lines += [r'\hline', r'\end{tabular}', r'\medskip',


              r'\end{table}']
    return '\n'.join(lines)


combined_marginal_effects_tex = make_combined_marginal_effects_table(
    marginal_effects_tables)
combined_path = AIC_TABLES_DIRECTORY / 'marginal_effects_all.tex'
combined_path.write_text(colorize(combined_marginal_effects_tex),
                         encoding='utf-8')
print()
print(combined_path)
print(combined_marginal_effects_tex)

=== Car crashes : average marginal effects (percentage points) ===
                                                                     Variation  No injury  Injury  Fatality
Variable                                                                                                   
Baseline predicted probabilities                                      baseline       3.59   95.71      0.69
The individual is a passenger                                   dummy (0 to 1)      13.86  -16.55      2.69
Second-party light vehicle                                      dummy (0 to 1)      10.57  -12.83      2.25
Second-party heavy vehicle                                      dummy (0 to 1)      -0.59   -7.01      7.60
Total number of vehicles and pedestrians involved in the crash   per unit (+1)       4.28   -5.10      0.82
The individual is a female                                      dummy (0 to 1)      -1.76    2.10     -0.34
Second-party turning right                                      dummy

## Framework comparison





In [ ]:


from latex_tables import colorize, make_model_comparison_table

SAMPLE_ORDER = ['Motorized vehicles', 'MMV', 'Pedestrians', 'Single-vehicle']

COMPARISON_TEX = RESULTS_ROOT / 'model_framework_comparison.tex'
COMPARISON_CSV = RESULTS_ROOT / 'model_framework_comparison.csv'
ESTIMATED_MODELS = [
    ('Motorized vehicles', 'Logistic regression', results_ml_motorized_vehicles,
     'results_constant_car'),
    ('Motorized vehicles', 'Ordered probit', results_car_probit,
     'results_constant_car'),
   # ('Car crashes', 'Ordered logit', results_car_logit),
    ('MMV', 'Binary logistic regression', results_logit_mmv, 'results_constant_mmv'),
   # ('MMV', 'Binary probit', results_mmv_probit, 'results_constant_mmv'),
  #  ('Pedestrian', 'Ordered logit', results_pedes),
    ('Pedestrians', 'Ordered probit', results_pedes_probit,
     'results_pedes_cst'),
    ('Pedestrians', 'Logistic regression', results_pedestrian_mnl,
     'results_pedes_cst_mnl'),
    ('Single-vehicle', 'Ordered probit', results_solo_2, 'results_cst_solo'),
   # ('Single-vehicle', 'Ordered logit', results_solo_logit),
    ('Single-vehicle', 'Logistic regression', results_solo_mnl, 'results_cst_solo_mnl'),
]




# `K` EXCLUT les constantes alternatives-specifiques et les seuils des modeles
# ordonnes : la colonne compte les parametres associes aux variables
# explicatives, comme les tableaux de statistiques de chaque modele. Ce sont eux
# qui distinguent deux specifications ; les constantes, tout modele les depense.
#
# ATTENTION : l'AIC et le BIC des colonnes voisines utilisent, eux, le K TOTAL,
# parce que c'est leur definition. Un lecteur qui recalculerait 2K - 2LL depuis
# la colonne K ne retrouverait pas l'AIC affiche -- d'ou la note de bas de
# tableau, qui le dit.
def _rho_bar_squared(log_likelihood, parameters, null_log_likelihood):

    if null_log_likelihood in (None, 0) or null_log_likelihood != null_log_likelihood:
        return float('nan')
    return 1 - (float(log_likelihood) - int(parameters)) / float(null_log_likelihood)


def _null_log_likelihood(name):

    results = globals().get(name)
    if results is None:

        return None
    return _fit_statistics(results)['LL']


def _fit_statistics(results):
    data = results.data

    def pick(*names):
        for name in names:
            value = getattr(data, name, None)
            if value is not None:
                return value
        return None

    return {'N': pick('sampleSize', 'sample_size'),
            'K': pick('nparam', 'number_of_parameters'),
            'LL': pick('logLike', 'final_log_likelihood'),
            'LL0': pick('nullLoglike', 'null_log_likelihood'),
            'AIC': pick('akaike', 'akaike_information_criterion'),
            'BIC': pick('bayesian', 'bayesian_information_criterion')}



MIXED_LOGIT_DIRECTORY = Path('mixed_model')


MIXED_LOGIT_MODELS = [
    ('Motorized vehicles', 'mixed_logit_car_crashes_panel',
     df_motorized_vehicles, 'results_constant_car'),
    ('MMV', 'mixed_logit_mmv_panel', df_mmv, 'results_constant_mmv'),
    ('Pedestrians', 'mixed_logit_pedestrian_panel', df_pedestrian,
     'results_pedes_cst_mnl'),
    ('Single-vehicle', 'mixed_logit_sinv_panel', df_sv, 'results_cst_solo_mnl'),
]


def _read_biogeme_html(path):
    """Statistiques d'ajustement lues dans un rapport HTML de Biogeme."""
    text = path.read_text(encoding='utf-8', errors='replace')

    def field(label):
        match = re.search(re.escape(label) + r'.{0,120}?<td[^>]*>(.*?)</td>',
                          text, re.S)
        if match is None:
            raise ValueError(f'{label!r} introuvable dans {path.name}')
        return float(re.sub(r'<[^>]+>', '', match.group(1)).replace(',', '').strip())

    # Uniquement le tableau des parametres estimes : le rapport contient aussi
    # une matrice de correlations dont chaque ligne commence par un nom de
    # parametre, et la compter reviendrait a additionner les memes noms 9 fois.
    start = text.find('<h1>Estimated parameters</h1>')
    block = text[start:text.find('</table>', start)] if start >= 0 else ''
    names = re.findall(r'<tr class=biostyle><td>([^<]+)</td>', block)

    return {'K': int(field('Number of estimated parameters')),
            'intercepts': _count_intercept_names(names),
            'groups': int(field('Sample size')),
            'LL': field('Final log likelihood'),
            'AIC': field('Akaike Information Criterion'),
            'draws': int(field('Number of draws'))}


def mixed_logit_rows(models=None, directory=MIXED_LOGIT_DIRECTORY):
    """Une ligne de comparaison par mixed logit trouve dans `directory`."""
    models = MIXED_LOGIT_MODELS if models is None else models
    records = []
    for sample, pattern, frame, baseline_name in models:
        matches = sorted(directory.glob(f'{pattern}*.html'))
        if not matches:
            print(f'  [{sample}] aucun fichier {pattern}*.html, ignore')
            continue
        path = matches[-1]          # le suffixe ~NN le plus eleve est le dernier run
        statistics = _read_biogeme_html(path)

        rows = len(frame)
        # Verification d'integrite : le nombre de groupes du fichier doit etre le
        # nombre d'accidents du cadre. S'il differe, le modele n'a pas ete estime
        # sur cet echantillon et son BIC recalculte serait faux.
        crashes = frame['Num_Acc'].nunique()
        if statistics['groups'] != crashes:
            print(f'  [{sample}] ATTENTION {path.name} : {statistics["groups"]} groupes '
                  f'contre {crashes} accidents dans le cadre courant -- '
                  f'echantillon different, BIC non recalcule')
            rows = None

        parameters, loglikelihood = statistics['K'], statistics['LL']
        bic = (math.log(rows) * parameters - 2 * loglikelihood
               if rows else float('nan'))
        records.append({
            'Sample': sample, 'Framework': 'Mixed logistic regression (crash panel)',
            'N': rows if rows else statistics['groups'],
            'K': parameters - statistics['intercepts'],
            'LL': loglikelihood,
            'AIC': statistics['AIC'], 'BIC': bic,
            'rho bar squared': _rho_bar_squared(
                loglikelihood, parameters, _null_log_likelihood(baseline_name)),
        })
        print(f'  [{sample}] {path.name} : K={parameters} LL={loglikelihood:.3f} '
              f'draws={statistics["draws"]}')
    return records


def build_comparison_table(models=None, sample_order=None,
                           include_mixed_logit=True):
    """DataFrame Sample / Framework / N / K / LL / AIC / BIC + les deltas.

    Les deltas sont mesures a l'interieur de chaque echantillon, contre le
    meilleur modele au sens du critere considere. Les lignes sont triees par
    echantillon (`sample_order`) puis par BIC croissant.
    """
    models = ESTIMATED_MODELS if models is None else models
    sample_order = SAMPLE_ORDER if sample_order is None else sample_order

    records = []
    for sample, framework, results, baseline_name in models:
        fit = _fit_statistics(results)
        records.append({'Sample': sample, 'Framework': framework,
                        'N': int(fit['N']),
                        'K': int(fit['K']) - _count_intercepts(results),
                        'LL': float(fit['LL']), 'AIC': float(fit['AIC']),
                        'BIC': float(fit['BIC']),
                        'rho bar squared': _rho_bar_squared(
                            fit['LL'], fit['K'],
                            _null_log_likelihood(baseline_name))})
    if include_mixed_logit:
        records.extend(mixed_logit_rows())
    table = pd.DataFrame.from_records(records)

    for criterion in ('AIC', 'BIC'):
        best = table.groupby('Sample')[criterion].transform('min')
        table[f'Delta {criterion}'] = table[criterion] - best

    # Ordre des echantillons impose ; un echantillon absent de la liste passe
    # a la fin plutot que de disparaitre.
    rank = {sample: position for position, sample in enumerate(sample_order)}
    table['_rank'] = table['Sample'].map(lambda s: rank.get(s, len(rank)))
    table = (table.sort_values(['_rank', 'AIC'])
                  .drop(columns='_rank')
                  .reset_index(drop=True))
    return table


comparison_table = build_comparison_table()
comparison_table.to_csv(COMPARISON_CSV, index=False)

comparison_tex = make_model_comparison_table(comparison_table,
                                             sample_order=SAMPLE_ORDER)
COMPARISON_TEX.write_text(colorize(comparison_tex), encoding='utf-8')

print(f'{COMPARISON_CSV}\n{COMPARISON_TEX}\n')
display(comparison_table.round(2))
print(comparison_tex)

  [Motorized vehicles] mixed_logit_car_crashes_panel~05.html : K=17 LL=-1450.161 draws=10000
  [MMV] ATTENTION mixed_logit_mmv_panel_identified.html : 595 groupes contre 685 accidents dans le cadre courant -- echantillon different, BIC non recalcule
  [MMV] mixed_logit_mmv_panel_identified.html : K=10 LL=-670.809 draws=10000
  [Pedestrians] ATTENTION mixed_logit_pedestrian_panel_identified~01.html : 1365 groupes contre 1367 accidents dans le cadre courant -- echantillon different, BIC non recalcule
  [Pedestrians] mixed_logit_pedestrian_panel_identified~01.html : K=9 LL=-1057.901 draws=10000
  [Single-vehicle] mixed_logit_sinv_panel~01.html : K=8 LL=-284.858 draws=5000
results/model_framework_comparison.csv
results/model_framework_comparison.tex



,Sample,Framework,N,K,LL,AIC,BIC,rho bar squared,Delta AIC,Delta BIC
0,Motorized vehicles,Logistic regression,9101,14,-1450.17,2932.33,3046.19,0.18,0.00,0.00
1,Motorized vehicles,Mixed logistic regression (crash panel),9101,15,-1450.16,2934.32,3055.30,0.18,1.99,9.11
2,Motorized vehicles,Ordered probit,9101,8,-1498.53,3017.07,3088.23,0.15,84.73,42.04
3,MMV,Binary logistic regression,1191,8,-670.81,1359.62,1405.36,0.13,0.00,0.00
4,MMV,Mixed logistic regression (crash panel),595,9,-670.81,1361.62,NaN,0.13,2.00,NaN
5,Pedestrians,Logistic regression,2790,7,-1057.90,2131.80,2179.27,0.44,0.00,0.00
6,Pedestrians,Mixed logistic regression (crash panel),1365,8,-1057.90,2133.80,NaN,0.44,2.00,NaN
7,Pedestrians,Ordered probit,2790,8,-1124.13,2268.26,2327.60,0.42,136.46,148.33
8,Single-vehicle,Logistic regression,2212,5,-284.86,583.72,623.63,0.11,0.00,0.00
9,Single-vehicle,Mixed logistic regression (crash panel),2212,6,-284.86,585.72,631.33,0.11,2.00,7.70


\begin{addedblock}
\begin{table}[H]
\begin{addedblock}
\centering
\small
\caption{Goodness-of-fit of the estimated models, by segments.}
\label{tab:model_comparison}
\begin{tabular}{l r r r r r}
\hline
Framework & N & K & LL & AIC & $\bar{\rho}^2$ \\
\hline
\multicolumn{6}{l}{\textit{Motorized vehicles}} \\
Logistic regression & 9101 & 14 & -1450.17 & \textbf{2932.33} & 0.177 \\
Mixed logistic regression (crash panel) & 9101 & 15 & -1450.16 & 2934.32 & 0.177 \\
Ordered probit & 9101 & 8 & -1498.53 & 3017.07 & 0.154 \\
\hline
\multicolumn{6}{l}{\textit{MMV}} \\
Binary logistic regression & 1191 & 8 & -670.81 & \textbf{1359.62} & 0.131 \\
Mixed logistic regression (crash panel) & 595 & 9 & -670.81 & 1361.62 & 0.130 \\
\hline
\multicolumn{6}{l}{\textit{Pedestrians}} \\
Logistic regression & 2790 & 7 & -1057.90 & \textbf{2131.80} & 0.439 \\
Mixed logistic regression (crash panel) & 1365 & 8 & -1057.90 & 2133.80 & 0.439 \\
Ordered probit & 2790 & 8 & -1124.13 & 2268.26 & 0.420 \\
\hline
\mu

## Poolability




In [ ]:


import numpy as np
from scipy.stats import chi2 as chi2_distribution, norm as normal_distribution

# Les variables retenues dans chaque specification estimee. A ajuster si tu
# modifies un modele : ce sont des noms de COLONNES, pas des noms de Beta.
SEGMENT_SPECIFICATIONS = {
    'Car crashes': [
        'age', 'Gender_Female', 'User category_Passenger',
        'number of involved vehicles', 'Point of impact_Back',
        'Intersection_No intersection', 'Maneuver_2_Turning right',
        'Lighting conditions_Night with street lightings on',
        'Accident location_On cycle facility',
    ],
    'MMV': [
        'age', 'Gender_Female', 'Surface condition_Wet', 'Maneuver_Swerving',
        'Maneuver_Turning left', 'Point of impact_Back', 'Gender_2_Female',
        'age_opposite_mean',
    ],
    'Pedestrian': [
        'age', 'Gender_Female', 'User category_Pedestrian',
        'Crossroad_Traffic lights', 'Intersection_No intersection',
        'Gender_2_Female', 'age_opposite_mean',
    ],
    'Single-vehicle': [
        'age', 'User category_Passenger', 'Long profile_Slope',
        'Number of passengers',
    ],
}

POOLABILITY_FRAMES = {
    'Car crashes': df_motorized_vehicles, 'MMV': df_mmv,
    'Pedestrian': df_pedestrian, 'Single-vehicle': df_sv,
}


# --- Le modele restreint ------------------------------------------------------
# Un seul modele, un seul coefficient par covariable, estime sur les quatre
# segments empiles : c'est le modele CONTRAINT du test de poolabilite, celui qui
# impose a chaque effet d'etre le meme dans tous les segments. Sa specification
# est l'UNION des quatre specifications retenues, chaque covariable n'entrant
# qu'une fois.
#
# La variable expliquee est la marge blesse-ou-tue contre indemne. C'est la seule
# reponse commune aux quatre segments depuis que les modeles MMV et pieton
# fusionnent le deces avec la blessure.
#
# Les constantes propres aux segments sont CONSERVEES. Sans elles le test
# confondrait deux choses : la difference des pentes, qui est la question posee,
# et la difference des taux de blessure entre segments, qui n'a rien a y voir.
# Le premier segment sert de reference et n'a donc pas de constante propre.

OUTCOME = 'outcome'            # 1 = indemne, 2 = blesse ou tue


def _biogeme_name(text, prefix):
    """Nom de parametre valide a partir d'un libelle quelconque."""
    body = ''.join(character if character.isalnum() else '_'
                   for character in str(text)).strip('_').lower()
    return f'{prefix}_{body}'


# L'union, dans l'ordre de premiere apparition : une covariable retenue par
# plusieurs segments ne doit entrer qu'une fois dans le modele poole.
POOLED_COVARIATES = list(dict.fromkeys(
    column for columns in SEGMENT_SPECIFICATIONS.values() for column in columns))

missing = {name: [column for column in POOLED_COVARIATES
                  if column not in frame.columns]
           for name, frame in POOLABILITY_FRAMES.items()}
missing = {name: columns for name, columns in missing.items() if columns}
if missing:
    raise KeyError(f'colonnes absentes de certains segments : {missing}')

# Les quatre segments empiles, avec une indicatrice par segment.
SEGMENT_DUMMIES = {name: _biogeme_name(name, 'segment')
                   for name in POOLABILITY_FRAMES}

pieces = []
for name, frame in POOLABILITY_FRAMES.items():
    piece = frame[POOLED_COVARIATES].copy()
    piece[OUTCOME] = (frame['severity'] > 1).astype(int) + 1
    for other in POOLABILITY_FRAMES:
        piece[SEGMENT_DUMMIES[other]] = int(other == name)
    pieces.append(piece)

pooled_data = pd.concat(pieces, ignore_index=True)
database_pooled = db.Database('poolability_restricted', pooled_data)


reference_segment, *other_segments = POOLABILITY_FRAMES

utility_restricted = Beta('asc_pooled', 0, None, None, 0)
for name in other_segments:
    utility_restricted += (Beta(_biogeme_name(name, 'asc'), 0, None, None, 0)
                           * Variable(SEGMENT_DUMMIES[name]))
for column in POOLED_COVARIATES:
    utility_restricted += (Beta(_biogeme_name(column, 'beta'), 0, None, None, 0)
                           * Variable(column))

logprob_restricted = models.loglogit({1: 0, 2: utility_restricted},
                                     {1: 1, 2: 1}, Variable(OUTCOME))

model_restricted = bio.BIOGEME(database_pooled, logprob_restricted,
                               generate_html=False, generate_pickle=False)
model_restricted.modelName = 'poolability_restricted'
results_restricted = model_restricted.estimate()

restricted_fit = _fit_statistics(results_restricted)
print(f'Modele restreint : reference = {reference_segment}')
print(f'  covariables : {len(POOLED_COVARIATES)}')
print(f'  N  = {int(restricted_fit["N"])}')
print(f'  K  = {int(restricted_fit["K"])}')
print(f'  LL = {float(restricted_fit["LL"]):.2f}')
results_restricted.get_estimated_parameters().round(4)


Modele restreint : reference = Car crashes
  covariables : 18
  N  = 15387
  K  = 22
  LL = -3170.07


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_mmv,-3.0804,0.1048,-29.4050,0.0000
asc_pedestrian,-4.6064,0.0994,-46.3382,0.0000
asc_pooled,3.8388,0.0980,39.1821,0.0000
asc_single_vehicle,-0.3228,0.2592,-1.2456,0.2129
beta_accident_location_on_cycle_facility,0.0332,0.0764,0.4343,0.6640
beta_age,0.0220,0.0021,10.2829,0.0000
beta_age_opposite_mean,-0.0159,0.0020,-8.0882,0.0000
beta_crossroad_traffic_lights,-0.0108,0.0880,-0.1233,0.9019
beta_gender_2_female,-0.7828,0.0712,-10.9959,0.0000
beta_gender_female,0.9608,0.0810,11.8601,0.0000


In [ ]:
# --- Les quatre modeles de segment, reestimes sur la marge blesse / indemne ---
# Le cote NON CONTRAINT du test, obtenu sans modele interagi : on estime le meme
# modele separement sur chaque segment et on somme les quatre vraisemblances.
# C'est le terme somme_s LL_s(beta_s) de l'equation.
#
# LA SPECIFICATION EST CELLE DU MODELE POOLE, l'union des covariables, et non
# celle retenue par chaque segment lors du developpement. C'est la condition de
# validite du test : le modele restreint doit etre un CAS PARTICULIER du non
# contraint, celui ou les quatre jeux de coefficients sont egaux. Avec les
# specifications propres, le modele poole dispose de covariables que les modeles
# de segment n'ont pas -- `Number of passengers` n'est retenue qu'en
# vehicule-seul, mais le poole lui donne un effet partout -- donc il peut fitter
# MIEUX que leur somme : le chi2 sort negatif et le test ne veut rien dire.
#
# La reestimation est necessaire : les modeles du manuscrit portent, pour deux
# segments, sur la severite a trois niveaux, et leurs vraisemblances ne sont donc
# pas comparables a celle du modele poole. Ici les quatre sont ramenes a la meme
# variable expliquee que lui, la marge blesse-ou-tue contre indemne.
#
# Chaque segment a sa propre constante, comme le modele poole a les siennes.

segment_rows = []
segment_results = {}

for name, frame in POOLABILITY_FRAMES.items():
    columns = list(POOLED_COVARIATES)

    data = frame[columns].copy()
    data[OUTCOME] = (frame['severity'] > 1).astype(int) + 1

    # Toute covariable n'est pas estimable dans tout segment, et l'union en
    # apporte forcement des inutilisables.
    #
    #   - constante dans le segment : `User category_Pedestrian` est nulle
    #     partout hors du segment pieton, son coefficient n'existe pas ;
    #   - separee de l'issue : une indicatrice dont toutes les observations a 1
    #     sont blessees n'a pas de maximum fini, le coefficient part a l'infini
    #     et emporte la vraisemblance avec lui.
    #
    # Le controle de separation ne vaut que pour les indicatrices : sur une
    # variable continue, `colonne > 0` serait souvent constante et la rejetterait
    # a tort.
    def _usable(column):
        values = data[column]
        if values.nunique() < 2:
            return 'constante dans ce segment'
        if set(values.dropna().unique()) <= {0, 1}:
            table = pd.crosstab(values > 0, data[OUTCOME])
            if table.size < 4 or table.to_numpy().min() < 1:
                return 'separee de l\'issue dans ce segment'
        return None

    for column in list(columns):
        reason = _usable(column)
        if reason:
            print(f'  [{name}] {column} : {reason}, ecartee')
            columns.remove(column)

    database_segment = db.Database(_biogeme_name(name, 'pool'), data)

    utility = Beta(_biogeme_name(name, 'asc'), 0, None, None, 0)
    for column in columns:
        utility += (Beta(_biogeme_name(f'{name}_{column}', 'beta'),
                         0, None, None, 0)
                    * Variable(column))

    logprob = models.loglogit({1: 0, 2: utility}, {1: 1, 2: 1}, Variable(OUTCOME))
    the_biogeme = bio.BIOGEME(database_segment, logprob,
                              generate_html=False, generate_pickle=False)
    the_biogeme.modelName = _biogeme_name(name, 'poolability')
    results = the_biogeme.estimate()

    segment_results[name] = results
    fit = _fit_statistics(results)
    segment_rows.append({'Segment': name, 'N': int(fit['N']),
                         'K': int(fit['K']), 'LL': float(fit['LL'])})

segment_fits = pd.DataFrame(segment_rows)
display(segment_fits.round(2))

# --- Le test ------------------------------------------------------------------
unrestricted_ll = float(segment_fits['LL'].sum())
unrestricted_k = int(segment_fits['K'].sum())
restricted_ll = float(restricted_fit['LL'])
restricted_k = int(restricted_fit['K'])

statistic = -2 * (restricted_ll - unrestricted_ll)
degrees = unrestricted_k - restricted_k
p_value = float(chi2_distribution.sf(statistic, degrees))

print()
print(f'restreint    : K = {restricted_k:3d}, LL = {restricted_ll:10.2f}')
print(f'non contraint: K = {unrestricted_k:3d}, LL = {unrestricted_ll:10.2f}'
      f'   (somme des 4 segments)')
print(f'chi2 = {statistic:.2f}, df = {degrees}, p = {p_value:.3g}')
print('->', 'les pentes different entre segments : la segmentation est justifiee.'
      if p_value < 0.05 else
      'pas de difference significative des pentes entre segments.')


  [Car crashes] User category_Pedestrian : constante dans ce segment, ecartee
  [MMV] User category_Pedestrian : constante dans ce segment, ecartee
  [Single-vehicle] Maneuver_2_Turning right : constante dans ce segment, ecartee
  [Single-vehicle] Maneuver_Turning left : separee de l'issue dans ce segment, ecartee
  [Single-vehicle] Gender_2_Female : constante dans ce segment, ecartee
  [Single-vehicle] age_opposite_mean : constante dans ce segment, ecartee
  [Single-vehicle] User category_Pedestrian : constante dans ce segment, ecartee


,Segment,N,K,LL
0,Car crashes,9101,18,-1201.24
1,MMV,1282,18,-691.71
2,Pedestrian,2792,19,-1049.82
3,Single-vehicle,2212,14,-98.45



restreint    : K =  22, LL =   -3170.07
non contraint: K =  69, LL =   -3041.23   (somme des 4 segments)
chi2 = 257.67, df = 47, p = 7.52e-31
-> les pentes different entre segments : la segmentation est justifiee.


## Sensitivity analysis: crashes with two parties only

A small share of the crashes involves more than two parties, and the database
does not say which opponent is the main one. Their second-party variables are
therefore coded by convention -- the mean over the opposing parties for a
continuous variable, "at least one opponent has this characteristic" for a
categorical one. The convention is defensible but it is a convention, and the
second-party effects are among the main findings of this study.

The four models are therefore re-estimated on the subsample where the coding is
unambiguous: the crashes with exactly two parties. Coefficients that hold are
not driven by the convention; coefficients that move should be read with the
caveat in mind. The single-vehicle segment has no opponent at all, so its
sample is unchanged and it serves as a control: any difference there would mean
the code is filtering something else than what it claims.


In [ ]:
# --- Sensitivity analysis: crashes with two parties only ----------------------
# Chaque modele est reestime tel quel -- meme specification, meme formule -- sur
# le sous-echantillon sans troisieme partie. On ne reconstruit rien : la
# vraisemblance `the_biogeme.log_like` est reutilisee sur une base filtree, donc
# aucun risque de reecrire une utilite differemment de l'originale.

THIRD_PARTY_PREFIX = 'vehicle_type_3_'


def two_party_mask(data):
    """Lignes dont l'accident ne compte que deux parties.

    Le critere est l'ABSENCE de troisieme partie, pas le nombre de vehicules :
    un pieton n'est pas un vehicule, et `number of involved vehicles` le
    compterait mal. Les indicatrices `vehicle_type_3_*` sont a zero partout
    quand il n'y a pas de troisieme partie ; a defaut on retombe sur le nombre
    de vehicules, qui coincide avec ce critere sur ce jeu de donnees.
    """
    columns = [column for column in data.columns
               if column.startswith(THIRD_PARTY_PREFIX)]
    if columns:
        return data[columns].sum(axis=1) == 0
    return data['number of involved vehicles'] <= 2


# Le modele a constantes seules de chaque segment, pour le rho-barre-deux.
# Resolu dans les globales : une cellule d'estimation non executee laisse le nom
# absent, et le tableau doit sortir quand meme, rho-barre en moins.
NULL_MODEL_NAMES = {
    'Car crashes': 'model_cst_car',
    'MMV': 'model_cst_mmv',
    'Pedestrian': 'model_cst_pedes_mnl',
    'Single-vehicle': 'model_cst_solo_mnl',
}


def _estimate_like(the_biogeme, data, name):
    """Reestime la formule de `the_biogeme` sur `data`, sans rien reecrire."""
    rebuilt = bio.BIOGEME(db.Database(name, data.reset_index(drop=True)),
                          the_biogeme.log_like,
                          number_of_draws=the_biogeme.number_of_draws,
                          generate_html=False, generate_pickle=False)
    rebuilt.modelName = name
    return rebuilt.estimate()


def _estimates(results):
    """{parametre: (valeur, t)} d'un resultat Biogeme."""
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)
    return {name: (float(record[column['value']]), float(record[column['t']]))
            for name, record in table.iterrows()}


sensitivity_results, sensitivity_tables, sensitivity_samples = {}, {}, []

for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    data = the_biogeme.database.data
    mask = two_party_mask(data)
    kept = data.loc[mask].reset_index(drop=True)

    print(f'=== {label} : {len(kept)} / {len(data)} observations '
          f'({100 * (1 - len(kept) / len(data)):.1f} % retirees) ===')

    restricted = _estimate_like(the_biogeme, kept, f'two_party_{segment}')
    sensitivity_results[label] = restricted

    full_estimates = _estimates(results)
    two_party_estimates = _estimates(restricted)

    rows = []
    for name, (value, t_statistic) in full_estimates.items():
        restricted_value, restricted_t = two_party_estimates.get(name,
                                                                 (float('nan'),) * 2)
        rows.append({'Parameter': _pretty(name),
                     'All crashes': value, 't': t_statistic,
                     'Two parties': restricted_value, 't (2p)': restricted_t,
                     'Change (%)': (100 * (restricted_value - value) / value
                                    if value else float('nan'))})
    table = pd.DataFrame.from_records(rows).set_index('Parameter')
    sensitivity_tables[label] = table
    display(table.round(3))

    full_fit, restricted_fit_2p = _fit_statistics(results), _fit_statistics(restricted)

    # Le rho-barre-deux de chaque echantillon se mesure contre SON PROPRE LL(c) :
    # le modele a constantes seules est reestime sur le sous-echantillon a deux
    # parties. Reutiliser le LL(c) de l'echantillon complet comparerait une
    # vraisemblance calculee sur 8 912 observations a une reference calculee sur
    # 9 450, et le rho ne voudrait rien dire.
    null_biogeme = globals().get(NULL_MODEL_NAMES.get(label, ''))
    if null_biogeme is None:
        print('  modele a constantes seules absent, rho-barre non calcule')
        rho_all = rho_two_party = float('nan')
    else:
        null_all = _estimate_like(null_biogeme, data, f'null_{segment}')
        null_two_party = _estimate_like(null_biogeme, kept,
                                        f'null_two_party_{segment}')
        rho_all = _rho_bar_squared(full_fit['LL'], full_fit['K'],
                                   _fit_statistics(null_all)['LL'])
        rho_two_party = _rho_bar_squared(restricted_fit_2p['LL'],
                                         restricted_fit_2p['K'],
                                         _fit_statistics(null_two_party)['LL'])

    sensitivity_samples.append({
        'Segment': label,
        'N (all)': int(full_fit['N']), 'N (two parties)': int(restricted_fit_2p['N']),
        'LL (all)': float(full_fit['LL']),
        'LL (two parties)': float(restricted_fit_2p['LL']),
        'rho bar squared (all)': rho_all,
        'rho bar squared (two parties)': rho_two_party,
    })
    print()

sensitivity_summary = pd.DataFrame.from_records(sensitivity_samples)
display(sensitivity_summary.round(2))


=== Car crashes : 8885 / 9101 observations (2.4 % retirees) ===


,All crashes,t,Two parties,t (2p),Change (%)
Parameter,,,,,
Crash occurred on a cycle facility (fatality),-0.890,-1.825,-0.890,-1.799,0.021
Age of the individual (years) (fatality),0.054,5.865,0.056,5.760,2.778
Age of the individual (years) (injury),0.016,3.462,0.018,3.725,15.047
The individual is a female (injury),0.630,4.668,0.644,4.587,2.137
Crash occurred away from an intersection (injury),0.249,2.003,0.241,1.863,-3.109
Crash occurred at night (with or without street lighting) (fatality),0.871,2.967,0.891,2.980,2.308
A second-party vehicle is turning right (fatality),1.091,3.643,1.167,3.867,6.996
The rider was not changing direction (injury),0.411,3.550,0.371,3.093,-9.723
Total number of vehicles and pedestrians involved in the crash (injury),-0.958,-4.692,-0.622,-47.649,-35.123



=== MMV : 1158 / 1191 observations (2.8 % retirees) ===


,All crashes,t,Two parties,t (2p),Change (%)
Parameter,,,,,
Mean age of the second-party individuals (years) (injury),-0.027,-5.623,-0.027,-5.567,0.440
Age of the individual (years) (injury),0.032,6.337,0.033,6.351,1.611
At least one second-party individual is a female (injury),-0.909,-6.496,-1.079,-7.408,18.657
The individual is a female (injury),1.077,6.399,1.097,6.432,1.906
The individual was swerving (injury),-0.502,-2.659,-0.485,-2.541,-3.233
The individual was turning left (injury),-1.356,-4.275,-1.323,-4.149,-2.405
Rear impact on the MMV (injury),-0.886,-4.188,-0.972,-4.533,9.723
Wet surface (injury),-0.542,-2.362,-0.565,-2.404,4.195
Alternative specific constant (injury),0.778,7.970,0.800,8.096,2.759



=== Pedestrian : 2685 / 2790 observations (3.8 % retirees) ===


,All crashes,t,Two parties,t (2p),Change (%)
Parameter,,,,,
Alternative specific constant,-0.804,-6.440,-0.774,-5.927,-3.724
Age of the individual (years) (injury),0.027,9.406,0.030,9.888,13.009
Mean age of the second-party individuals (years) (injury),-0.019,-7.004,-0.019,-7.080,4.125
Crash occurred at a signalized intersection (injury),0.330,2.206,0.343,2.190,4.058
At least one second-party individual is a female (injury),-1.095,-9.617,-1.166,-9.782,6.483
The individual is a female (injury),0.993,8.540,1.089,9.037,9.619
Crash occurred away from an intersection (injury),0.351,2.690,0.331,2.426,-5.564
The individual is a pedestrian (injury),2.284,18.465,2.361,18.003,3.371



=== Single-vehicle : 2212 / 2212 observations (0.0 % retirees) ===


,All crashes,t,Two parties,t (2p),Change (%)
Parameter,,,,,
Alternative specific constants,-0.572,-1.779,-0.572,-1.779,-0.002
Alternative specific constants,4.837,19.740,4.837,19.740,-0.000
Age of the individual (years) (fatality),0.071,4.784,0.071,4.785,0.002
Age of the individual (years) (injury),0.027,2.098,0.027,2.098,-0.001
Crash occurred on a slope (fatality),1.012,2.251,1.012,2.251,-0.001
The individual is a passenger (injury),-2.640,-7.317,-2.640,-7.317,0.000
The rider was on an e-PMD,-0.713,-2.526,-0.713,-2.526,-0.001


,Segment,N (all),N (two parties),LL (all),LL (two parties),rho bar squared (all),rho bar squared (two parties)
0,Car crashes,9101,8885,-1450.17,-1352.15,0.18,0.18
1,MMV,1191,1158,-670.81,-643.99,0.13,0.14
2,Pedestrian,2790,2685,-1057.90,-972.95,0.44,0.46
3,Single-vehicle,2212,2212,-284.86,-284.86,0.11,0.11


## Manuscript tables

The four estimation tables of the manuscript, written straight from the estimated
models -- same layout, same wording, same significance stars -- so that
re-estimating a model updates the thesis instead of asking for a manual copy.
Each row maps a sentence to the Biogeme parameters that carry it; a parameter
shared by two utilities is printed in both columns and flagged by a footnote,
and any estimated parameter missing from the mapping is appended at the end of
the table rather than silently dropped. Files go to `results/manuscript/`.


In [ ]:
# --- Manuscript tables --------------------------------------------------------
# The four estimation tables of the manuscript, written straight from the
# estimated models: same layout, same wording, same significance stars, so that
# re-estimating a model updates the thesis instead of asking for a manual copy.
#
# A row maps a human sentence to the Biogeme parameters that carry it, one per
# column of the table ('Injury' / 'Fatality' for the MNL, a single column for the
# binary and ordered models). `None` prints the dash used when a variable does
# not enter that utility, and the same parameter in two columns is a coefficient
# constrained to be equal -- flagged by a footnote, as in the manuscript.
#
# Any estimated parameter missing from the spec is appended in a final section
# rather than silently dropped: the printed table always accounts for K.

import math

def first_defined(*names):
    """Le premier de ces noms qui existe dans le notebook.

    Les modeles evoluent (l'ordered logit pietonnier a laisse place a l'ordered
    probit) : on nomme les candidats par ordre de preference plutot que de
    referencer une variable qui peut avoir disparu.
    """
    for name in names:
        if name in globals():
            return globals()[name]
    raise NameError(f'aucun de ces resultats n\'est defini : {", ".join(names)}')


MANUSCRIPT_DIRECTORY = RESULTS_ROOT / 'manuscript'
MANUSCRIPT_DIRECTORY.mkdir(parents=True, exist_ok=True)

SIGNIFICANCE = [(0.01, '***'), (0.05, '**'), (0.10, '*')]

THRESHOLD_LABELS = [r"Threshold between `No injury' and `Injury' ($\tau_1$)",
                    r"Threshold between `Injury' and `Fatality' ($\tau_2$)"]


def _number(value, digits=3):
    """3 chiffres significatifs, comme dans le manuscrit (0.0159, -1.32, 6.04)."""
    return f'{float(value):.{digits}g}'


def _stars(p_value):
    if p_value is None or p_value != p_value:
        return ''
    for threshold, mark in SIGNIFICANCE:
        if float(p_value) < threshold:
            return mark
    return ''


def _estimates(results):
    """{nom: (valeur, ecart-type robuste, p-value robuste)}."""
    table = results.get_estimated_parameters()
    column = _parameter_columns(table)
    return {name: (float(record[column['value']]),
                   float(record[column['std']]),
                   float(record[column['p']]))
            for name, record in table.iterrows()}


def _threshold_rows(results, estimates):
    """(libelle, valeur, se, p) pour tau_1 et tau_2 = tau_1 + diff.

    Biogeme estime le premier seuil et l'ecart au suivant ; le second seuil est
    donc une combinaison lineaire, dont l'ecart-type se calcule exactement a
    partir de la covariance robuste.
    """
    names = [name for name in estimates if name.startswith('tau')]
    base = [name for name in names if 'diff' not in name]
    if not base:
        return []

    first = base[0]
    rows = [(THRESHOLD_LABELS[0], *estimates[first])]

    differences = [name for name in names if name.startswith(f'{first}_diff')]
    if differences:
        second = differences[0]
        covariance = results.get_robust_var_covar()
        value = estimates[first][0] + estimates[second][0]
        variance = (float(covariance.loc[first, first])
                    + float(covariance.loc[second, second])
                    + 2 * float(covariance.loc[first, second]))
        std_error = math.sqrt(variance)
        p_value = 2 * (1 - norm.cdf(abs(value / std_error)))
        rows.append((THRESHOLD_LABELS[1], value, std_error, p_value))
    return rows


# Seuil au-dela duquel un coefficient retenu est signale en note, et la note
# elle-meme. Un coefficient dont le t robuste n'est pas significatif a 5 % a pu
# etre conserve parce que le test du rapport de vraisemblance, lui, conclut a
# une amelioration de l'ajustement : c'est la procedure de selection decrite
# dans le manuscrit, et il faut le dire sous le tableau plutot que laisser
# croire a un oubli.
RETAINED_ABOVE_ALPHA = 0.05

# Les coefficients de second tiers portent un appel de note renvoyant a
# l'analyse de sensibilite sur les accidents a deux parties. C'est la section du
# tableau qui les designe, pas une liste de noms a tenir a jour : toute variable
# rangee sous ce titre est concernee, par construction.
SECOND_PARTY_SECTION = 'Second-party characteristics'
SECOND_PARTY_MARK = r'$^{\dagger}$'
SECOND_PARTY_NOTE = (
    r'$^{\dagger}$ For these variables, an estimation restricted to '
    'crashes involving exactly two parties yields coefficients of the same sign, magnitude '
    'and significance.')
RETAINED_ABOVE_ALPHA_NOTE = (
    'A few variables whose robust $t$-test p-value exceeds 0.05 are retained '
    'because the likelihood ratio test shows a significant improvement in '
    'model fit.')


def make_manuscript_table(results, specification, baseline=None):
    """Le longtable d'estimation et la petite table de statistiques."""
    estimates = _estimates(results)
    columns = specification['columns']
    width = 1 + 2 * len(columns)
    used = set()

    def cells(parameters):
        row = []
        for column in columns:
            name = parameters.get(column)
            if name is None or name not in estimates:
                row += ['-', '-']
                continue
            value, std_error, p_value = estimates[name]
            used.add(name)
            row += [f'{_number(value)} {_stars(p_value)}'.strip(),
                    _number(std_error)]
        return row

    body = []
    second_party_rows = []

    for label, value, std_error, p_value in _threshold_rows(results, estimates):
        used.update(name for name in estimates if name.startswith('tau'))
        body.append(_row([label, f'{_number(value)} {_stars(p_value)}'.strip(),
                          _number(std_error)]))

    for section, rows in specification['sections']:
        for row in rows:
            if not (isinstance(row, tuple) and len(row) == 2
                    and isinstance(row[1], dict)):
                raise ValueError(
                    f"section '{section}' : ligne malformee, attendu "
                    f"(libelle, {{colonne: parametre}}), recu {row!r}")
        printed = [(label, parameters) for label, parameters in rows
                   if any(name in estimates for name in parameters.values()
                          if name is not None)]
        if not printed:
            continue
        # Un titre vide n'imprime pas d'en-tete : les constantes alternatives-
        # specifiques ouvrent le tableau et leur libelle de ligne dit deja ce
        # qu'elles sont, un intertitre par-dessus ne ferait que le repeter.
        if section:
            body += [r'\\', r'{\textbf{' + section + r'}}\\']
        mark = SECOND_PARTY_MARK if section == SECOND_PARTY_SECTION else ''
        if mark:
            second_party_rows.append(section)
        for label, parameters in printed:
            body.append(_row([label + mark] + cells(parameters)))

    # Filet de securite : rien ne disparait du tableau.
    forgotten = [name for name in estimates
                 if name not in used and not name.startswith('tau')]
    if forgotten:
        body += [r'\\', r'{\textbf{Other estimated parameters}}\\']
        for name in forgotten:
            body.append(_row([_escape(name)]
                             + cells({columns[0]: name})))

    if len(columns) == 1:
        alignment = 'p{7cm}p{2cm}p{1.5cm}'
        header = [_row([r'\textbf{Variable}', r'\textit{Estimates}',
                        r'\textit{SE}'])]
    else:
        alignment = 'p{8cm}' + 'p{2cm}p{1.5cm}' * len(columns)
        header = [
            _row([r'\multirow{2}{*}{\textbf{Variable}}']
                 + [r'\multicolumn{2}{c}{\textbf{' + column + '}}'
                    for column in columns]),
            _row([''] + [item for _ in columns
                         for item in (r'\textit{Estimates}', '{SE}')]),
        ]

    significance = (r'\footnotesize Level of significance: * $p<0.10$, '
                    r'** $p<0.05$, *** $p<0.01$.')
    notes = [rf'\multicolumn{{{width}}}{{l}}{{SE: robust standard error}} \\',
             rf'\multicolumn{{{width}}}{{l}}{{' + significance + r'} \\']

    # Note conditionnelle : elle n'apparait que si le tableau contient
    # effectivement un coefficient non significatif au seuil de 5 %. L'ecrire
    # systematiquement ferait chercher au lecteur une ligne qui n'existe pas
    # dans ce tableau-la ; la calculer sur les parametres REELLEMENT imprimes
    # (`used`) evite aussi de la declencher pour un coefficient ecarte.
    if second_party_rows:
        notes.append(rf'\multicolumn{{{width}}}{{p{{\linewidth}}}}{{\footnotesize '
                     + SECOND_PARTY_NOTE + r'} \\')

    if any(estimates[name][2] > RETAINED_ABOVE_ALPHA for name in used
           if name in estimates):
        notes.append(rf'\multicolumn{{{width}}}{{p{{\linewidth}}}}{{\footnotesize '
                     + RETAINED_ABOVE_ALPHA_NOTE + r'} \\')

    for note in specification.get('notes', []):
        notes.append(rf'\multicolumn{{{width}}}{{p{{\linewidth}}}}{{\footnotesize '
                     + note + r'} \\')

    lines = [
        r'\begin{small}',
        rf'\begin{{longtable}}{{{alignment}}}',
        rf'\caption{{{specification["caption"]}}} \label{{{specification["label"]}}} \\',
        r'\hline', *header, r'\hline', r'\endfirsthead',
        r'{\textit{(continued)}} \\',
        r'\hline', *header, r'\hline', r'\endhead',
        rf'\hline \multicolumn{{{width}}}{{r}}{{\textit{{Continued on next page}}}} \\',
        r'\endfoot',
        r'\hline', r'\endlastfoot',
        *body,
        r'\\', *notes,
        r'\end{longtable}',
        r'\end{small}',
    ]

    fit = _fit_statistics(results)
    baseline_fit = _fit_statistics(baseline) if baseline is not None else None
    null_log_likelihood = (baseline_fit['LL'] if baseline_fit is not None
                           else fit['LL0'])

    # `K` EXCLUT les constantes alternatives-specifiques et les seuils tau : le
    # libelle du tableau annonce les parametres associes aux VARIABLES
    # EXPLICATIVES, et une constante n'en est pas une. Le rho-barre-deux qui
    # suit penalise donc, lui aussi, les seuls coefficients ajoutes par-dessus
    # le modele de reference -- ce qui est coherent, sa reference etant
    # justement LL(c), le modele reduit a ces constantes.
    #
    # Le compte des intercepts est pris sur le modele a constantes seules
    # lui-meme plutot que par filtrage des noms : cela vaut aussi bien pour un
    # MNL (une constante par alternative) que pour un modele ordonne (les taus).
    # Sans modele de reference, on retombe sur les noms de parametres.
    #
    # ATTENTION : l'AIC et le BIC utilisent, eux, le K TOTAL -- c'est leur
    # definition. Un lecteur qui recalculerait 2K - 2LL depuis cette ligne ne
    # retrouverait pas l'AIC du tableau de comparaison des cadres.
    if baseline_fit is not None:
        estimated_constants = int(baseline_fit['K'])
    else:
        estimated_constants = _count_intercepts(results)
    reported_parameters = int(fit['K']) - estimated_constants

    # Observations ecartees : `full_sample` donne la taille du segment complet,
    # `dropped_note` la raison. La note n'apparait que si l'ecart est non nul,
    # pour qu'un tableau dont l'echantillon est intact reste muet.
    full_sample = specification.get('full_sample')
    full_sample = full_sample() if callable(full_sample) else full_sample
    dropped = int(full_sample) - int(fit['N']) if full_sample else 0
    dropped_mark = r'$^{\dagger}$' if dropped > 0 else ''

    statistics = [
        r'\begin{small}', r'\begin{table}[h!]', r'\centering',
        rf'\caption{{{specification["stats_caption"]}}}',
        rf'\label{{{specification["stats_label"]}}}',
        r'\begin{tabular}{l c}', r'\hline',
        _row([r'\textbf{Number of observations}' + dropped_mark,
              f'{int(fit["N"]):,}'.replace(',', '{,}')]),
        _row([r'\textbf{Number of estimated parameters associated with the explanatory variables $K$}',
              f'{reported_parameters}']),
    ]
    if null_log_likelihood is not None:
        statistics.append(_row([r'\textbf{$LL(c)$}',
                                f'{float(null_log_likelihood):,.0f}'.replace(',', '{,}')]))
    statistics.append(_row([r'\textbf{$LL(\hat{\beta})$}',
                            f'{float(fit["LL"]):,.0f}'.replace(',', '{,}')]))
    if null_log_likelihood is not None:
        rho_bar = (1 - (float(fit['LL']) - reported_parameters)
                   / float(null_log_likelihood))
        statistics.append(_row(
            [r'\textbf{$\bar{\rho}^2 = 1 - \dfrac{LL(\hat{\beta}) - K}{LL(c)}$}',
             f'{rho_bar:.3f}']))
    statistics.append(r'\hline')
    if dropped > 0:
        note = specification.get(
            'dropped_note',
            'observations excluded from the estimation sample.')
        statistics.append(
            r'\multicolumn{2}{p{\dimexpr\linewidth-2\tabcolsep}}{\footnotesize '
            rf'$^{{\dagger}}$ {dropped} ' + note + r'} \\')
    statistics += [r'\end{tabular}', r'\end{table}', r'\end{small}']

    return '\n'.join(lines) + '\n\n' + '\n'.join(statistics)




manuscript_tex = {}
for name, specification in MANUSCRIPT_SPECIFICATIONS.items():
    try:
        results, baseline = specification['results']()
        tex = make_manuscript_table(results, specification, baseline=baseline)
    except Exception as error:
        print(f'{name:16s} : tableau non produit -- '
              f'{type(error).__name__}: {error}')
        continue
    path = MANUSCRIPT_DIRECTORY / specification['file']
    path.write_text(colorize(tex) + '\n', encoding='utf-8')
    manuscript_tex[name] = tex
    print(f'{name:16s} -> {path}')

Car crashes      -> results/manuscript/model_car.tex
MMV              -> results/manuscript/model_mmv.tex
Pedestrian (MNL) -> results/manuscript/model_ped_mnl.tex
Single-vehicle (MNL) -> results/manuscript/model_solo_mnl.tex


In [ ]:
def get_results(file_path):
    """Load a Biogeme `bioResults` object back from its pickle file."""
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return res.bioResults(data)




## Out-of sample validation of the models

In [ ]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_motorized_vehicles)
validationData_mmv= create_validation_data(df_mmv_identified)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian_identified)

In [ ]:
# Validate the model with the validation data for car
validation_results_car = model_car.validate(results_ml_motorized_vehicles, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_motorized_vehicles, validationData_car)


# Initialize variables to store log-likelihoods

loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (car)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_car += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (cars)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_car += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (cars)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





Log likelihood for 2129 validation data on car (slide 1): -343.469969336575
Log likelihood for 1361 validation data on car (slide 2): -227.84386444347672
Log likelihood for 2129 validation data on car (constant model, slide 1): -400.5337894989798
Log likelihood for 1361 validation data on car (constant model, slide 2): -249.84570033178488
Rho-square for the validation data on car (slide 1): 0.14246942869360624
Rho-square for the validation data on car (slide 2): 0.08806169511458728


In [ ]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





Log likelihood for 342 validation data on mmv (slide 1): -193.0752209181369
Log likelihood for 81 validation data on mmv (slide 2): -43.0057370706338
Log likelihood for 342 validation data on mmv (constant model, slide 1): -223.45986907013997
Log likelihood for 81 validation data on mmv (constant model, slide 2): -51.251836143922844
Rho-square for the validation data on mmv (slide 1): 0.13597362371346366
Rho-square for the validation data on mmv (slide 2): 0.16089372974136495


In [ ]:

# Validate the model with the validation data for mmv
validation_results_pedes = model_pedes_probit.validate(results_pedestrian_mnl, validationData_pedestrian)
validation_results_pedes_cst = model_cst_pedes.validate(results_pedes_cst_mnl, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_pedes= 0
loglike_constant_pedes = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_pedes):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_pedes += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_pedes_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_pedes += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on pedes (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_pedes)):
    validation_loglike = validation_results_pedes[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_pedes_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on pedes (slide {i+1}): {rho_square}')




Log likelihood for 709 validation data on mmv (slide 1): -302.6670516162159
Log likelihood for 223 validation data on mmv (slide 2): -91.0512599145923
Log likelihood for 709 validation data on pedes (constant model, slide 1): -490.1706057174865
Log likelihood for 223 validation data on pedes (constant model, slide 2): -152.0017925166846
Rho-square for the validation data on pedes (slide 1): 0.38252712813493295
Rho-square for the validation data on pedes (slide 2): 0.4009856172939672


In [ ]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


Log likelihood for 422 validation data on mmv (slide 1): -72.9697903678165
Log likelihood for 222 validation data on mmv (slide 2): -70.59692156660606
Log likelihood for 422 validation data on mmv (constant model, slide 1): -83.7974453870486
Log likelihood for 222 validation data on mmv (constant model, slide 2): -76.58265527365617
Rho-square for the validation data on mmv (slide 1): 0.1292122327741697
Rho-square for the validation data on mmv (slide 2): 0.07816043574959675


### Validation tables

The same figures as the loops above, assembled into one table per segment and
written to `results/tables/`: the size of each validation set, the
log-likelihood of the model and of its constants-only counterpart, and the
out-of-sample $1 - LL(\hat{\beta})/LL(c)$.


In [ ]:
# --- Out-of-sample validation tables ------------------------------------------
# One table per segment, in the format used in the manuscript: one column per
# validation split, and for each of them the size of the validation set, the
# log-likelihood of the model, that of the constants-only model, and the
# out-of-sample rho-square 1 - LL(beta) / LL(c).
#
# `validate()` re-estimates the model on the estimation part of each split and
# applies it to the validation part, so this cell re-estimates 2 x 2 models per
# segment: it is slow. Called directly on the Biogeme object, it writes its
# intermediate files in the working directory rather than in `results/`.

VALIDATION_TABLES_DIRECTORY = RESULTS_ROOT / 'tables'
VALIDATION_TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)

# `create_validation_data` builds the splits in this order: (1) estimate on
# 2019-2022, validate on 2023, then (2) estimate on Paris, validate on Lyon.
# The table shows them in the order of the manuscript.
VALIDATION_SLIDES = [(1, 'Lyon vs. Paris'),
                     (0, 'Year 2023 vs. Years 2019--2022')]

SCRATCH_PATTERNS = ('__*.iter', '*.html', '*.pickle')


@contextmanager
def _quiet_validation(*biogeme_objects):
    """Estimate without leaving HTML, pickle or iteration files behind.

    `validate()` re-estimates the model once per split, and each of those fits
    writes `<modelName>.html`, `<modelName>.pickle` and `__<modelName>.iter`
    into the working directory -- which is how the repository root filled up
    with `__*_val_est_*.iter`. The three flags below switch that off; the
    sweep afterwards removes whatever a Biogeme version writes regardless,
    and it only touches files that did not exist before the call.
    """
    flags = ('generate_html', 'generate_pickle', 'save_iterations')
    saved = [(obj, {flag: getattr(obj, flag, None) for flag in flags})
             for obj in biogeme_objects]
    before = {path for pattern in SCRATCH_PATTERNS
              for path in Path.cwd().glob(pattern)}
    try:
        for obj in biogeme_objects:
            for flag in flags:
                if hasattr(obj, flag):
                    setattr(obj, flag, False)
        yield
    finally:
        for obj, previous in saved:
            for flag, value in previous.items():
                if value is not None:
                    setattr(obj, flag, value)
        created = {path for pattern in SCRATCH_PATTERNS
                   for path in Path.cwd().glob(pattern)} - before
        for path in created:
            path.unlink(missing_ok=True)
        if created:
            print(f'  {len(created)} fichier(s) intermediaire(s) supprime(s)')


VALIDATION_ROWS = ['Number of obs. in the validation set',
                   r'\textbf{$LL(\hat{\beta})$}',
                   r'\textbf{$LL(c)$}',
                   r'\textbf{$1 - \frac{LL(\hat{\beta})}{LL(c)}$}']


def validation_table(the_biogeme, results, the_biogeme_constant,
                     results_constant, validation_data, slides=None):
    """Une colonne par decoupage, les quatre lignes du tableau de validation."""
    slides = VALIDATION_SLIDES if slides is None else slides

    with _quiet_validation(the_biogeme, the_biogeme_constant):
        model_slides = the_biogeme.validate(results, validation_data)
        constant_slides = the_biogeme_constant.validate(results_constant,
                                                        validation_data)

    columns = {}
    for position, label in slides:
        model = model_slides[position]
        constant = constant_slides[position]
        log_likelihood = float(model['Loglikelihood'].sum())
        log_likelihood_constant = float(constant['Loglikelihood'].sum())
        columns[label] = [
            model.shape[0],
            log_likelihood,
            log_likelihood_constant,
            1 - log_likelihood / log_likelihood_constant,
        ]
    return pd.DataFrame(columns, index=VALIDATION_ROWS)


def make_validation_table(table, caption=None, label=None,
                          first_width='5.5cm', width='5cm'):
    """Tableau LaTeX de validation, au format du manuscrit."""
    centred = r'>{\centering\arraybackslash}p'
    alignment = (f'{centred}{{{first_width}}}'
                 + f'{centred}{{{width}}}' * len(table.columns))

    lines = [
        r'\begin{table}[H]', r'\small ', r'\centering     ',
        rf'\caption{{{caption}}}' if caption else r'\caption{}',
        rf'\label{{{label}}}' if label else '',
        '',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline',
        '',
        ' & ' + ' & '.join(str(column) for column in table.columns) + r'  \\',
        r'\hline',
    ]

    formats = ['{:.0f}', '{:.0f}', '{:.0f}', '{:.3f}']
    for (name, record), template in zip(table.iterrows(), formats):
        cells = [template.format(float(value)) for value in record]
        lines.append(f'{name} & ' + ' & '.join(cells) + r'\\')

    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


# (libelle, modele, resultats, modele a constantes, ses resultats,
#  donnees de validation, nom de fichier, legende)
VALIDATION_MODELS = [
    ('Car crashes', model_car, results_ml_motorized_vehicles,
     model_cst_car, results_constant_car, validationData_car, 'val_car',
     "rider's injury severity in crashes with a motor vehicle"),
    ('MMV', model_mmv, results_logit_mmv,
     model_cst_mmv, results_constant_mmv, validationData_mmv, 'val_mmv',
     "rider's injury severity in crashes with a micromobility vehicle"),
    # Pedestrian and single-vehicle are validated on their MNL, each against a
    # constants-only model of its own framework and its own dependent variable.
    # This matters for the pedestrian: its MNL is a binary logit on `harmed`,
    # so referencing it against the three-level ordered-probit baseline would
    # inflate the rho-square.
    ('Pedestrian', model_pedestrian_mnl, results_pedestrian_mnl,
     model_cst_pedes_mnl, results_pedes_cst_mnl, validationData_pedestrian,
     'val_pedes', "injury severity in crashes involving a pedestrian"),
    ('Single-vehicle', model_solo_mnl, results_solo_mnl,
     model_cst_solo_mnl, results_cst_solo_mnl, validationData_sv_2, 'val_solo',
     "rider's injury severity in single-vehicle crashes"),
]

validation_tables, validation_tex = {}, {}
for (name, the_biogeme, results, the_biogeme_constant,
     results_constant, validation_data, stem, subject) in VALIDATION_MODELS:
    print(f'=== {name} ===')
    try:
        table = validation_table(the_biogeme, results,
                                 the_biogeme_constant, results_constant,
                                 validation_data)
    except Exception as error:
        print(f'  validation impossible : {type(error).__name__}: {error}\n')
        continue

    tex = make_validation_table(
        table,
        caption=('Results of the out-of-sample validation for the model '
                 f'predicting {subject}'),
        label=f'tab:{stem}',
    )
    (VALIDATION_TABLES_DIRECTORY / f'{stem}.tex').write_text(
        colorize(tex), encoding='utf-8')
    validation_tables[name], validation_tex[name] = table, tex

    print(table.round(3).to_string())
    print(f'  -> {VALIDATION_TABLES_DIRECTORY / f"{stem}.tex"}\n')

print(next(iter(validation_tex.values()), ''))


# --- The four validations in a single table -----------------------------------
# One block of columns per split, four sub-columns per block -- one per model --
# and the four statistics as rows. More compact than four separate tables, and
# it lets the reader compare the models on the same split at a glance.
MODEL_NUMBERS = {name: f'({position})'
                 for position, name in enumerate(validation_tables, start=1)}


def make_combined_validation_table(
    tables,
    caption=('Out-of-sample validation of the four models. '
             '(1) crashes with a motor vehicle, (2) crashes with another '
             'micromobility vehicle, (3) crashes involving a pedestrian, '
             '(4) single-vehicle crashes.'),
    label='tab:validation_all',
):
    models = list(tables)
    splits = [label_text for _, label_text in VALIDATION_SLIDES]
    rows = VALIDATION_ROWS
    width = 1 + len(splits) * len(models)

    # `{|c}` reproduit le filet vertical du preambule dans l'en-tete groupe.
    group_header = _row([''] + [r'\multicolumn{' + str(len(models))
                                + r'}{|c}{\textbf{' + split + '}}'
                                for split in splits])
    rules = ''.join(rf'\cline{{{2 + position * len(models)}-'
                    rf'{1 + (position + 1) * len(models)}}}'
                    for position in range(len(splits)))
    model_header = _row([r'\textbf{Statistic}']
                        + [MODEL_NUMBERS[name] for _ in splits for name in models])
    alignment = ('p{5.5cm}'
                 + ('|' + 'r' * len(models)) * len(splits))

    lines = [
        r'\begin{table}[H]', r'\small', r'\centering',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}',
        rf'\begin{{tabular}}{{{alignment}}}',
        r'\hline', group_header, rules, model_header, r'\hline',
    ]

    formats = ['{:.0f}', '{:.0f}', '{:.0f}', '{:.3f}']
    for row, template in zip(rows, formats):
        cells = []
        for split in splits:
            for name in models:
                table = tables[name]
                value = table[split][row] if split in table.columns else None
                cells.append('' if value is None else template.format(float(value)))
        lines.append(_row([row] + cells))

    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


combined_validation_tex = make_combined_validation_table(validation_tables)
combined_validation_path = VALIDATION_TABLES_DIRECTORY / 'val_all.tex'
combined_validation_path.write_text(colorize(combined_validation_tex) + '\n',
                                    encoding='utf-8')
print()
print(combined_validation_path)
print(combined_validation_tex)

=== Car crashes ===
  4 fichier(s) intermediaire(s) supprime(s)
                                              Lyon vs. Paris  Year 2023 vs. Years 2019--2022
Number of obs. in the validation set                1361.000                        2129.000
\textbf{$LL(\hat{\beta})$}                          -227.844                        -343.470
\textbf{$LL(c)$}                                    -249.846                        -400.534
\textbf{$1 - \frac{LL(\hat{\beta})}{LL(c)}$}           0.088                           0.142
  -> results/tables/val_car.tex

=== MMV ===
  4 fichier(s) intermediaire(s) supprime(s)
                                              Lyon vs. Paris  Year 2023 vs. Years 2019--2022
Number of obs. in the validation set                  81.000                         342.000
\textbf{$LL(\hat{\beta})$}                           -43.006                        -193.075
\textbf{$LL(c)$}                                     -51.252                        -223.460
\textbf{$1